# ポケカAIバトル公式上位対戦を眺めるEDA

Kaggle公式の上位対戦履歴を、カード画像と日本語名つきで見やすく整理します。

まず知りたいのは、上位対戦で「誰が勝っていて、どんなカードがよく見えて、どの試合を見直すべきか」です。
このノートブックでは次をざっと確認できます。

- 公式データセットの日付、件数、スコアの流れ
- チーム別の勝敗と、よく出る対面
- 試合の長さ、短すぎる試合、長引いた試合
- よく出るアクション内容とターン帯
- よく見えるカードの日本語名と公式カード画像
- 勝者側/敗者側に偏って見えるカード
- 次にリプレイで見直したい対戦候補

Kaggle上では、公式の対戦環境データ、公式の対戦一覧、日別の対戦ログを入力に追加して実行します。

- `pokemon-tcg-ai-battle`
- `pokemon-tcg-ai-battle-episodes-index`
- `pokemon-tcg-ai-battle-episodes-YYYY-MM-DD`

カード名は公式のカード一覧CSV、カード画像は公式PDFから必要分だけ取り出します。




In [ ]:
import ast
import base64
import collections
import csv
import gc
import glob
import json
import math
import os
import re
import statistics
import subprocess
import sys
from html import escape
import zipfile
from pathlib import Path

RUNNING_IN_IPYTHON = False
try:
    get_ipython  # type: ignore[name-defined]
    RUNNING_IN_IPYTHON = get_ipython() is not None
except Exception:
    RUNNING_IN_IPYTHON = False

if not RUNNING_IN_IPYTHON:
    import matplotlib
    matplotlib.use("Agg")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.ft2font import FT2Font
import seaborn as sns


def setup_japanese_matplotlib_font():
    preferred = [
        "Noto Sans CJK JP",
        "Noto Sans CJK SC",
        "Noto Sans CJK TC",
        "Noto Sans JP",
        "IPAexGothic",
        "IPAGothic",
        "Hiragino Sans",
        "Yu Gothic",
    ]
    available = {f.name for f in fm.fontManager.ttflist}
    for name in preferred:
        if name in available:
            plt.rcParams["font.family"] = name
            return name
    if Path("/kaggle/input").exists():
        try:
            print("Noto CJKフォントをインストールします。日本語名と一部CJKチーム名の豆腐化を防ぎます。")
            subprocess.run(["apt-get", "update", "-qq"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            subprocess.run(["apt-get", "install", "-y", "-qq", "fonts-noto-cjk"], check=False, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
            fm._load_fontmanager(try_read_cache=False)
            available = {f.name for f in fm.fontManager.ttflist}
            for name in preferred:
                if name in available:
                    plt.rcParams["font.family"] = name
                    return name
        except Exception as exc:
            print("Noto CJKフォントのインストールをスキップしました:", exc)
        try:
            print("日本語フォントが見つからないため、japanize-matplotlibをインストールします。")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "japanize-matplotlib"])
            import japanize_matplotlib  # noqa: F401
            fm._load_fontmanager(try_read_cache=False)
            available = {f.name for f in fm.fontManager.ttflist}
            for name in preferred:
                if name in available:
                    plt.rcParams["font.family"] = name
                    return name
            plt.rcParams["font.family"] = "IPAexGothic"
            return "IPAexGothic"
        except Exception as exc:
            print("日本語フォントのインストールに失敗しました。図中の日本語が文字化けする可能性があります:", exc)
    return plt.rcParams.get("font.family", ["DejaVu Sans"])[0]


JAPANESE_FONT = setup_japanese_matplotlib_font()
plt.rcParams["axes.unicode_minus"] = False
try:
    PLOT_FONT_FACE = FT2Font(fm.findfont(JAPANESE_FONT))
except Exception:
    PLOT_FONT_FACE = None

HAS_PLOT = True
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="deep",
    rc={
        "figure.facecolor": "white",
        "axes.facecolor": "#fbfbfd",
        "axes.edgecolor": "#d7dce2",
        "axes.labelcolor": "#2f3542",
        "axes.titleweight": "bold",
        "grid.color": "#e8ecf1",
        "grid.linewidth": 0.8,
        "font.size": 10,
        "font.family": JAPANESE_FONT,
        "font.sans-serif": [JAPANESE_FONT, "IPAexGothic", "Noto Sans CJK JP", "Hiragino Sans", "Yu Gothic", "DejaVu Sans", "Arial"],
        "axes.unicode_minus": False,
    },
)
PALETTE = {
    "blue": "#4C72B0",
    "orange": "#DD8452",
    "green": "#55A868",
    "red": "#C44E52",
    "purple": "#8172B2",
    "gray": "#8C8C8C",
    "pink": "#CC79A7",
    "gold": "#E69F00",
}
ACTION_FAMILY_JA = {
    "attack": "攻撃",
    "play_card": "カード使用",
    "attach_energy": "エネ付け",
    "retreat_switch": "にげる/交代",
    "evolve": "進化",
    "bench": "ベンチ",
    "choose_select": "選択",
    "pass_end": "終了",
    "none": "なし",
    "other": "その他",
}
SELECT_CONTEXT_JA = {
    0: "通常行動",
    1: "手札/場から選択",
    2: "任意選択",
    3: "効果対象選択",
    4: "相手側の場を選択",
    5: "山札から選択",
    7: "山札からカード選択",
    8: "効果で手札から選択",
    13: "相手ポケモン選択",
    14: "ダメカン/効果対象選択",
    21: "自分の場を選択",
    22: "自分の手札/場を選択",
    26: "エネルギー選択",
    30: "エネルギーコスト選択",
    37: "進化先/対象選択",
    38: "番号選択",
    41: "先攻/後攻選択",
    43: "特性/効果の選択",
}
SELECT_TYPE_JA = {
    0: "通常行動候補",
    1: "カード/対象候補",
    2: "エネルギー候補",
    4: "エネルギー支払い候補",
    7: "進化候補",
    8: "番号候補",
    9: "二択候補",
}
OPTION_TYPE_JA = {
    0: "番号",
    1: "はい/先攻",
    2: "いいえ/後攻",
    3: "カード",
    5: "エネルギー",
    6: "支払いエネルギー",
    7: "カード使用/場に出す",
    8: "手札カード使用/エネ付け",
    9: "進化先",
    12: "番を終える",
    13: "ワザ",
    14: "番を終える",
}
AREA_JA = {
    1: "山札",
    2: "手札",
    3: "トラッシュ",
    4: "バトル場",
    5: "ベンチ",
    6: "サイド",
    8: "ロスト/別領域",
    12: "付いているカード",
    14: "スタジアム/場",
}
LOG_TYPE_JA = {
    0: "ゲーム進行",
    1: "初手/たね確認",
    2: "ターン開始",
    3: "ターン終了",
    4: "カード効果",
    5: "状態更新",
    6: "ダメージ/きぜつ",
    7: "カード移動",
    8: "山札/サーチ",
    10: "エネルギー",
    11: "ワザ",
    16: "勝敗/報酬",
    22: "選択処理",
}
PLOT_DIR = Path("/kaggle/working/ptcg_official_top_episodes_eda_ja_plots") if Path("/kaggle/working").exists() else Path("ptcg_official_top_episodes_eda_ja_plots")
PLOT_DIR.mkdir(parents=True, exist_ok=True)

def polish_axes(ax):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.tick_params(axis="both", colors="#444444")
    return ax

def safe_plot_label(value, limit=34):
    text = str(value)
    fallback_chars = {
        "馬": "Ma",
        "賽": "Sai",
    }
    parts = []
    for ch in text:
        if not ch.isprintable():
            parts.append("?")
        elif ch in fallback_chars:
            parts.append(fallback_chars[ch])
        elif PLOT_FONT_FACE is not None and ord(ch) > 127 and PLOT_FONT_FACE.get_char_index(ord(ch)) == 0:
            parts.append("?")
        else:
            parts.append(ch)
    cleaned = "".join(parts)
    if len(cleaned) > limit:
        cleaned = cleaned[: limit - 1] + "…"
    return cleaned


def area_label(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return "なし/不明"
    if str(value).lower() in {"none", "nan", ""}:
        return "なし/不明"
    try:
        numeric = int(float(value))
    except Exception:
        return f"エリア{value}"
    return AREA_JA.get(numeric, f"エリア{numeric}")


def html_img(path, width=116):
    path = Path(path)
    if not path.exists():
        return ""
    mime = "image/png" if path.suffix.lower() == ".png" else "image/jpeg"
    encoded = base64.b64encode(path.read_bytes()).decode("ascii")
    return f'<img src="data:{mime};base64,{encoded}" style="width:{width}px;border-radius:8px;box-shadow:0 8px 18px rgba(15,23,42,.16);">'

def finish_plot(name, title=None, subtitle=None):
    fig = plt.gcf()
    ax = plt.gca()
    polish_axes(ax)
    if title:
        fig.suptitle(title, x=0.02, y=1.03, ha="left", fontsize=15, fontweight="bold", color="#24292f")
    if subtitle:
        fig.text(0.02, 0.985, subtitle, ha="left", va="top", fontsize=9, color="#5f6875")
    plt.tight_layout(rect=[0, 0, 1, 0.88])
    path = PLOT_DIR / name
    plt.savefig(path, dpi=180, bbox_inches="tight", facecolor="white")
    print("プロット保存:", path)
    if RUNNING_IN_IPYTHON:
        plt.show()
    else:
        plt.close()

pd.set_option("display.max_columns", 120)
pd.set_option("display.max_colwidth", 120)
try:
    display
except NameError:
    def display(value):
        print(value.head(20) if hasattr(value, "head") else value)


JP_COLUMN_LABELS = {
    "episode_id": "対戦ID",
    "date": "日付",
    "path": "パス",
    "steps": "ステップ数",
    "seed": "seed",
    "team_0": "チーム0",
    "team_1": "チーム1",
    "reward_0": "報酬0",
    "reward_1": "報酬1",
    "status_0": "状態0",
    "status_1": "状態1",
    "player_index": "プレイヤー番号",
    "team": "チーム",
    "opponent": "相手",
    "reward": "報酬",
    "is_win": "勝ち",
    "is_loss": "負け",
    "status": "状態",
    "episode_steps": "対戦ステップ数",
    "games": "試合数",
    "wins": "勝ち数",
    "losses": "負け数",
    "mean_reward": "平均報酬",
    "mean_episode_steps": "平均ステップ数",
    "win_rate": "勝率",
    "action_family": "アクション分類",
    "action": "アクション",
    "action_ids": "アクションID",
    "action_cards_ja": "アクション値の読み方",
    "resolved_action_ja": "実際に選んだ内容",
    "action_kind_ja": "行動カテゴリ",
    "action_value_label_ja": "アクション値",
    "select_context": "選択コンテキストID",
    "select_context_ja": "選択コンテキスト",
    "select_type": "選択タイプID",
    "select_type_ja": "選択タイプ",
    "select_option_count": "選択肢数",
    "select_summary_ja": "選択肢の概要",
    "effect_card_ja": "効果元カード",
    "context_card_ja": "文脈カード",
    "count": "件数",
    "card_id": "カードID",
    "action_id": "アクションID",
    "action_label_ja": "アクション値の表示",
    "winner_count": "勝者側件数",
    "loser_count": "敗者側件数",
    "display_name_ja": "日本語名",
    "name_en": "英語名",
    "observations": "観測数",
    "episodes": "見えた対戦数",
    "winner_obs_share": "勝者側観測率",
    "board_key_ja": "盤面条件",
    "board_stage_ja": "ターン帯",
    "board_pair_ja": "盤面→選択",
    "own_active_ja": "自分バトル場",
    "opp_active_ja": "相手バトル場",
    "own_bench_count": "自分ベンチ数",
    "opp_bench_count": "相手ベンチ数",
    "own_hand_count": "自分手札枚数",
    "own_prize_count": "自分サイド枚数",
    "opp_prize_count": "相手サイド枚数",
    "selected_share": "選択率",
    "winner_share": "勝者側比率",
    "confidence_note": "読み方",
    "official_image_path": "公式画像パス",
    "reason": "理由",
    "type": "イベント種別",
    "type_ja": "イベント種別",
    "fromArea": "移動元エリア",
    "toArea": "移動先エリア",
    "term": "検索語",
    "log": "ログ",
}


def display_ja(df, columns=None, head=None):
    out = df.copy()
    if columns is not None:
        out = out.loc[:, [c for c in columns if c in out.columns]]
    if head is not None:
        out = out.head(head)
    display(out.rename(columns=JP_COLUMN_LABELS))




## 設定

既定では、入力に追加した公式Top episodeリプレイを全件読みます。
軽く動作確認したい時だけ、環境変数 `PTCG_EDA_MAX_EPISODES=500` のように上限を指定します。
このあと、manifest上の件数、実ファイル内の件数、今回読んだ件数を並べて確認します。

日付指定を空にすると全日付、日付文字列を入れるとその日だけを見ます。




In [ ]:
MAX_EPISODES_RAW = os.environ.get("PTCG_EDA_MAX_EPISODES", "").strip()
MAX_EPISODES = int(MAX_EPISODES_RAW) if MAX_EPISODES_RAW else None
DATE_FILTER = os.environ.get("PTCG_EDA_DATE_FILTER") or None
MIN_ACTION_COUNT = 5
TOP_N = 30
RANDOM_SEED = 42

np.random.seed(RANDOM_SEED)




## 公式データセットの探索

想定している公式データセットです。

- `pokemon-tcg-ai-battle-episodes-index`
- `pokemon-tcg-ai-battle-episodes-YYYY-MM-DD`

日別データは、展開済みフォルダでも圧縮ファイルでも読めるようにしています。




In [ ]:
KAGGLE_INPUT = Path("/kaggle/input")


def find_local_root():
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "competitions/pokemon-tcg-ai-battle").exists():
            return p
    return Path.cwd()


LOCAL_ROOT = find_local_root()
LOCAL_ARTIFACT = LOCAL_ROOT / "competitions/pokemon-tcg-ai-battle/reports/artifacts/daily_top_episodes"
if not LOCAL_ARTIFACT.exists():
    LOCAL_ARTIFACT = LOCAL_ROOT / "reports/artifacts/daily_top_episodes"


def existing_paths(*paths):
    return [p for p in paths if p.exists()]


def find_manifest():
    candidates = []
    if KAGGLE_INPUT.exists():
        candidates += list(KAGGLE_INPUT.glob("pokemon-tcg-ai-battle-episodes-index/**/manifest.csv"))
        candidates += list(KAGGLE_INPUT.glob("**/manifest.csv"))
    candidates += existing_paths(LOCAL_ARTIFACT / "manifest.csv")
    seen = []
    for p in candidates:
        if p not in seen:
            seen.append(p)
    for p in seen:
        try:
            cols = set(pd.read_csv(p, nrows=1).columns)
        except Exception:
            continue
        if {"date", "episode_count"}.issubset(cols):
            return p
    return None

manifest_path = find_manifest()
manifest_path




In [ ]:
if manifest_path is not None:
    manifest = pd.read_csv(manifest_path)
else:
    manifest = pd.DataFrame(columns=["date", "daily_dataset_slug", "daily_dataset_url", "episode_count", "total_bytes", "top_avg_score", "median_avg_score"])

for col in ["episode_count", "total_bytes"]:
    if col in manifest:
        manifest[col] = pd.to_numeric(manifest[col], errors="coerce").astype("Int64")
for col in ["top_avg_score", "median_avg_score"]:
    if col in manifest:
        manifest[col] = pd.to_numeric(manifest[col], errors="coerce")

manifest




In [ ]:
if len(manifest):
    display_ja(manifest.tail(10))
    print("最新日付:", manifest["date"].max())
    print("manifest行数:", len(manifest))
    print("manifest上の合計バイト数:", int(manifest["total_bytes"].fillna(0).sum()))
else:
    print("manifest.csvが見つかりません。公式index datasetを入力に追加するか、ローカル成果物を使ってください。")




## 公式カードマスターと日本語カード画像

カードIDだけでは中身が分からないため、公式入力のカード一覧を読みます。

- 日本語名の一覧
- 英語名の一覧
- 日本語カード画像PDFから必要分だけ抽出

Kaggle上では、公式の対戦環境データを入力に追加しておけば同じ処理で動きます。




In [ ]:
def find_input_file(names):
    roots = [KAGGLE_INPUT, LOCAL_ROOT / "competitions/pokemon-tcg-ai-battle/input", LOCAL_ROOT / "input", Path.cwd()]
    for root in roots:
        if not root.exists():
            continue
        for name in names:
            direct = root / name
            if direct.exists():
                return direct
        for name in names:
            matches = list(root.rglob(name))
            if matches:
                return matches[0]
    return None


JP_CARD_CSV = find_input_file(["JP_Card_Data.csv"])
EN_CARD_CSV = find_input_file(["EN_Card_Data.csv"])
JP_CARD_PDF = find_input_file(["Card_ID List_JP.pdf", "Card_ID%20List_JP.pdf"])


def load_card_master():
    frames = []
    if JP_CARD_CSV and JP_CARD_CSV.exists():
        jp = pd.read_csv(JP_CARD_CSV)
        jp = jp.rename(columns={
            "カード ID": "card_id",
            "カード名": "name_ja",
            "エキスパンションマーク": "expansion_ja",
            "コレクション番号": "collection_no_ja",
            "カテゴリ": "category_ja",
            "タイプ": "type_ja",
        })
        keep = [c for c in ["card_id", "name_ja", "expansion_ja", "collection_no_ja", "category_ja", "type_ja"] if c in jp]
        frames.append(jp[keep])
    if EN_CARD_CSV and EN_CARD_CSV.exists():
        en = pd.read_csv(EN_CARD_CSV)
        en = en.rename(columns={
            "Card ID": "card_id",
            "Card Name": "name_en",
            "Expansion": "expansion_en",
            "Collection No.": "collection_no_en",
            "Category": "category_en",
            "Type": "type_en",
        })
        keep = [c for c in ["card_id", "name_en", "expansion_en", "collection_no_en", "category_en", "type_en"] if c in en]
        frames.append(en[keep])
    if not frames:
        return pd.DataFrame(columns=["card_id", "name_ja", "name_en"])
    master = frames[0]
    for frame in frames[1:]:
        master = master.merge(frame, on="card_id", how="outer")
    master["card_id"] = pd.to_numeric(master["card_id"], errors="coerce").astype("Int64")
    master = master.dropna(subset=["card_id"]).drop_duplicates("card_id").sort_values("card_id").reset_index(drop=True)
    return master


card_master = load_card_master()
card_meta = {
    int(row.card_id): row._asdict()
    for row in card_master.itertuples(index=False)
    if pd.notna(row.card_id)
}


def parse_card_id(value):
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return None
    match = re.search(r"card_id:(\d+)", str(value))
    if match:
        return int(match.group(1))
    if str(value).strip().isdigit():
        return int(str(value).strip())
    return None


def card_display_name(value, limit=42):
    card_id = parse_card_id(value)
    if card_id is not None and card_id in card_meta:
        meta = card_meta[card_id]
        name = meta.get("name_ja") or meta.get("name_en") or f"カードID {card_id}"
        suffix = f"#{card_id}"
        return safe_plot_label(f"{name} ({suffix})", limit)
    return safe_plot_label(value, limit)


def card_name_by_id(card_id, limit=40):
    try:
        card_id = int(card_id)
    except Exception:
        return ""
    if card_id == 0:
        return "値0（カードIDではない）"
    meta = card_meta.get(card_id, {})
    name = meta.get("name_ja") or meta.get("name_en")
    if not name:
        return f"ID {card_id}"
    return safe_plot_label(f"{card_id} {name}", limit)


def action_value_label(value, select_obj=None, limit=40):
    try:
        value = int(value)
    except Exception:
        return str(value)
    options = []
    if isinstance(select_obj, dict) and isinstance(select_obj.get("option"), list):
        options = select_obj.get("option")
    if options and 0 <= value < len(options):
        return f"候補{value + 1}番目（選択肢番号）"
    return card_name_by_id(value, limit=limit)


def card_obj_display(obj, limit=46):
    if not isinstance(obj, dict):
        return ""
    card_id = obj.get("id")
    if card_id is None:
        return ""
    base = card_name_by_id(card_id, limit=limit)
    serial = obj.get("serial")
    if serial is not None:
        return safe_plot_label(f"{base} / serial {serial}", limit)
    return base


def zone_cards_from_option(option, cur, select_obj):
    if not isinstance(option, dict):
        return []
    area = option.get("area")
    player_idx = option.get("playerIndex")
    if area is None and option.get("type") in (7, 9) and "index" in option:
        area = 2
        player_idx = cur.get("yourIndex") if isinstance(cur, dict) else player_idx
    if area is not None and player_idx is None and isinstance(cur, dict):
        player_idx = cur.get("yourIndex")
    players = cur.get("players") if isinstance(cur, dict) else None
    player = None
    if isinstance(players, list) and isinstance(player_idx, (int, np.integer)) and 0 <= int(player_idx) < len(players):
        player = players[int(player_idx)]
    if area == 1:
        deck = select_obj.get("deck") if isinstance(select_obj, dict) else None
        return deck if isinstance(deck, list) else []
    if not isinstance(player, dict):
        return []
    zone_key = {
        2: "hand",
        3: "discard",
        4: "active",
        5: "bench",
        6: "prize",
    }.get(area)
    cards = player.get(zone_key) if zone_key else None
    return cards if isinstance(cards, list) else []


def card_from_option(option, cur, select_obj):
    cards = zone_cards_from_option(option, cur, select_obj)
    idx = option.get("index") if isinstance(option, dict) else None
    if isinstance(idx, (int, np.integer)) and 0 <= int(idx) < len(cards):
        return cards[int(idx)]
    return None


def resolved_option_display(option, cur, select_obj):
    if not isinstance(option, dict):
        return ""
    card = card_from_option(option, cur, select_obj)
    if card:
        prefix = OPTION_TYPE_JA.get(option.get("type"), "候補")
        area = AREA_JA.get(option.get("area"), "手札" if option.get("type") in (7, 9) and "area" not in option else f"エリア{option.get('area')}")
        return safe_plot_label(f"{prefix}: {card_obj_display(card, 54)} ({area})", 70)
    if option.get("type") in (12, 14):
        return "番を終える"
    if "attackId" in option:
        players = cur.get("players") if isinstance(cur, dict) else None
        your_idx = cur.get("yourIndex") if isinstance(cur, dict) else None
        active = None
        if isinstance(players, list) and isinstance(your_idx, (int, np.integer)) and 0 <= int(your_idx) < len(players):
            active_cards = players[int(your_idx)].get("active") if isinstance(players[int(your_idx)], dict) else None
            if isinstance(active_cards, list) and active_cards:
                active = active_cards[0]
        active_label = card_obj_display(active, 42) if active else "バトル場"
        return f"ワザ: {active_label} / ワザID {option.get('attackId')}"
    if "number" in option:
        return f"番号{option.get('number')}を選択"
    if option.get("type") in (1, 2) and "area" not in option:
        return OPTION_TYPE_JA.get(option.get("type"), option_display(option))
    return option_display(option)


def parse_action_ids(text):
    try:
        value = ast.literal_eval(text) if isinstance(text, str) else text
    except Exception:
        return []
    if isinstance(value, list):
        return [int(x) for x in value if isinstance(x, (int, np.integer))]
    if isinstance(value, (int, np.integer)):
        return [int(value)]
    return []


def resolved_action_display(action_text, select_obj=None, cur=None, limit_ids=12):
    ids = parse_action_ids(action_text)
    options = select_obj.get("option") if isinstance(select_obj, dict) and isinstance(select_obj.get("option"), list) else []
    # Opening/setup rows can contain a whole deck list while a select prompt is present.
    # If the first value points into select.option, treat the array as option selections
    # plus occasional scalar parameters. Otherwise treat it as card-id payload.
    looks_like_option_payload = bool(options) and ids and 0 <= ids[0] < len(options)
    if looks_like_option_payload:
        labels = [
            resolved_option_display(options[i], cur or {}, select_obj) if 0 <= i < len(options) else f"追加値 {i}"
            for i in ids[:limit_ids]
        ]
    else:
        labels = [card_name_by_id(i, 38) for i in ids[:limit_ids]]
    if len(ids) > limit_ids:
        labels.append(f"ほか{len(ids) - limit_ids}件")
    return " / ".join(labels)


def action_kind_from_resolved(text):
    text = "" if pd.isna(text) else str(text)
    if not text:
        return "不明"
    if "番を終える" in text:
        return "番を終える"
    if text.startswith("ワザ:") or "ワザID" in text:
        return "ワザ"
    if "カード使用/場に出す" in text:
        return "カード使用/場に出す"
    if "エネルギー" in text:
        return "エネルギー選択/付け替え"
    if "進化先" in text:
        return "進化"
    if "サイド" in text:
        return "サイド選択"
    if text.startswith("カード:"):
        return "カード選択"
    if "はい/先攻" in text or "いいえ/後攻" in text:
        return "先攻/後攻・二択"
    if "追加値" in text:
        return "補助値"
    return "その他"


def action_ids_display(action_text, select_obj=None, limit_ids=12):
    ids = parse_action_ids(action_text)
    labels = [action_value_label(i, select_obj, 34) for i in ids[:limit_ids]]
    if len(ids) > limit_ids:
        labels.append(f"ほか{len(ids) - limit_ids}件")
    return " / ".join(labels)


def option_display(option):
    if not isinstance(option, dict):
        return ""
    parts = [OPTION_TYPE_JA.get(option.get("type"), f"候補type {option.get('type')}")]
    if "area" in option:
        parts.append(AREA_JA.get(option.get("area"), f"エリア{option.get('area')}"))
    if "playerIndex" in option:
        parts.append(f"P{option.get('playerIndex')}")
    if "index" in option:
        parts.append(f"idx{option.get('index')}")
    if "attackId" in option:
        parts.append(f"ワザ{option.get('attackId')}")
    if "energyIndex" in option:
        parts.append(f"エネ{option.get('energyIndex')}")
    if "number" in option:
        parts.append(f"番号{option.get('number')}")
    if "count" in option:
        parts.append(f"{option.get('count')}枚")
    return " ".join(parts)


def select_summary(select_obj, limit_options=5):
    if not isinstance(select_obj, dict):
        return {}
    options = select_obj.get("option")
    options = options if isinstance(options, list) else []
    pieces = [option_display(o) for o in options[:limit_options]]
    pieces = [p for p in pieces if p]
    if len(options) > limit_options:
        pieces.append(f"ほか{len(options) - limit_options}候補")
    return {
        "select_context": select_obj.get("context"),
        "select_context_ja": SELECT_CONTEXT_JA.get(select_obj.get("context"), f"選択context {select_obj.get('context')}"),
        "select_type": select_obj.get("type"),
        "select_type_ja": SELECT_TYPE_JA.get(select_obj.get("type"), f"選択type {select_obj.get('type')}"),
        "select_option_count": len(options),
        "select_summary_ja": " / ".join(pieces),
        "effect_card_ja": card_obj_display(select_obj.get("effect")),
        "context_card_ja": card_obj_display(select_obj.get("contextCard")),
    }


def player_state_from_current(cur, player_idx):
    players = cur.get("players") if isinstance(cur, dict) else None
    if not isinstance(players, list) or not isinstance(player_idx, (int, np.integer)) or not (0 <= int(player_idx) < len(players)):
        return {}
    player = players[int(player_idx)]
    return player if isinstance(player, dict) else {}


def first_card_label_from_zone(player, zone_key):
    cards = player.get(zone_key) if isinstance(player, dict) else None
    if isinstance(cards, list) and cards:
        return card_obj_display(cards[0], 46)
    return "なし"


def visible_board_summary(cur, player_idx):
    if not isinstance(cur, dict):
        return {}
    try:
        player_idx = int(player_idx)
    except Exception:
        player_idx = cur.get("yourIndex", 0)
    opp_idx = 1 - player_idx if player_idx in (0, 1) else None
    own = player_state_from_current(cur, player_idx)
    opp = player_state_from_current(cur, opp_idx) if opp_idx is not None else {}
    own_active = first_card_label_from_zone(own, "active")
    opp_active = first_card_label_from_zone(opp, "active")
    own_bench_count = safe_len(own.get("bench"))
    opp_bench_count = safe_len(opp.get("bench"))
    own_hand_count = safe_len(own.get("hand"))
    own_prize_count = safe_len(own.get("prize"))
    opp_prize_count = safe_len(opp.get("prize"))
    turn = cur.get("turn")
    if pd.isna(turn):
        turn_label = "ターン不明"
    else:
        try:
            turn_value = int(float(turn))
            if turn_value <= 1:
                turn_label = "序盤0-1"
            elif turn_value <= 3:
                turn_label = "序盤2-3"
            elif turn_value <= 6:
                turn_label = "中盤4-6"
            elif turn_value <= 10:
                turn_label = "中盤7-10"
            else:
                turn_label = "終盤11+"
        except Exception:
            turn_label = "ターン不明"
    board_key = (
        f"{turn_label} / 自:{safe_plot_label(own_active, 18)} / "
        f"相手:{safe_plot_label(opp_active, 18)} / Bench{own_bench_count}"
    )
    return {
        "board_stage_ja": turn_label,
        "own_active_ja": own_active,
        "opp_active_ja": opp_active,
        "own_bench_count": own_bench_count,
        "opp_bench_count": opp_bench_count,
        "own_hand_count": own_hand_count,
        "own_prize_count": own_prize_count,
        "opp_prize_count": opp_prize_count,
        "board_key_ja": board_key,
    }


def ensure_fitz():
    try:
        import fitz
        return fitz
    except Exception as first_exc:
        if not KAGGLE_INPUT.exists():
            print("PyMuPDF(fitz)がないため、PDF画像抽出をスキップします:", first_exc)
            return None
        try:
            print("PyMuPDF(fitz)がないため、pymupdfをインストールします。カード画像ソースは公式PDFのみです。")
            subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pymupdf"])
            import fitz
            return fitz
        except Exception as second_exc:
            print("pymupdfのインストール/読み込みに失敗したため、PDF画像抽出をスキップします:", second_exc)
            return None


def official_pdf_card_page_map(pdf_path):
    fitz = ensure_fitz()
    if fitz is None:
        return {}
    doc = fitz.open(pdf_path)
    target_pages = []
    for page_no in range(doc.page_count):
        links = [
            link for link in doc.load_page(page_no).get_links()
            if link.get("kind") == fitz.LINK_GOTO
            and isinstance(link.get("page"), int)
            and int(link["page"]) > page_no
        ]
        if len(links) <= 1:
            continue
        links.sort(key=lambda link: (round(float(link["from"].y0), 3), round(float(link["from"].x0), 3)))
        target_pages.extend(int(link["page"]) for link in links)
    deduped = list(dict.fromkeys(target_pages))
    return {card_id: page_no for card_id, page_no in enumerate(deduped, start=1)}


CARD_IMAGE_DIR = Path("/kaggle/working/official_card_images_ja") if Path("/kaggle/working").exists() else Path("official_card_images_ja")
CARD_IMAGE_DIR.mkdir(parents=True, exist_ok=True)
PDF_PAGE_MAP = official_pdf_card_page_map(JP_CARD_PDF) if JP_CARD_PDF else {}


def extract_official_card_images(card_ids, out_dir=CARD_IMAGE_DIR, limit=36):
    if not JP_CARD_PDF or not PDF_PAGE_MAP:
        return {}
    fitz = ensure_fitz()
    if fitz is None:
        return {}
    out_dir.mkdir(parents=True, exist_ok=True)
    doc = fitz.open(JP_CARD_PDF)
    found = {}
    for card_id in list(dict.fromkeys(int(c) for c in card_ids if pd.notna(c)))[:limit]:
        page_no = PDF_PAGE_MAP.get(card_id)
        if page_no is None:
            continue
        meta = card_meta.get(card_id, {})
        raw_name = str(meta.get("name_ja") or f"card_{card_id}")
        safe_name = re.sub(r"[^\wぁ-んァ-ヶ一-龠ー【】]+", "_", raw_name).strip("_")
        out_path = out_dir / f"{card_id:04d}_{safe_name}.png"
        if not out_path.exists():
            images = doc.load_page(page_no).get_images(full=True)
            if not images:
                continue
            pix = fitz.Pixmap(doc, images[0][0])
            if pix.n - pix.alpha > 3:
                pix = fitz.Pixmap(fitz.csRGB, pix)
            pix.save(out_path)
        found[card_id] = str(out_path)
    return found


print("日本語カードCSV:", JP_CARD_CSV)
print("英語カードCSV:", EN_CARD_CSV)
print("日本語カードPDF:", JP_CARD_PDF)
print("カードマスター行数:", len(card_master))
print("PDFページ対応数:", len(PDF_PAGE_MAP))
display_ja(card_master, head=10)




## リプレイファイルの探索

対戦JSONはKaggleの対戦環境で使われるリプレイ形式です。

- 対戦全体の情報、報酬、状態、設定、時系列ステップ
- 各ステップの中に、プレイヤーごとの記録
- 各プレイヤー記録の中に、行動、観測、報酬、状態、補足情報

相手の手札などは非公開になりやすいので、ここでは公開盤面、選択アクション、ログ、勝敗を中心に見ます。




In [ ]:
def infer_date_from_path(path: Path):
    text = str(path)
    m = re.search(r"(20\d\d-\d\d-\d\d)", text)
    return m.group(1) if m else None


def discover_json_paths():
    paths = []
    if KAGGLE_INPUT.exists():
        for p in KAGGLE_INPUT.glob("**/*.json"):
            if p.name.endswith(".json") and p.name[:-5].isdigit():
                paths.append(p)
    paths += list((LOCAL_ARTIFACT / "daily_json").glob("**/*.json"))
    paths = sorted(set(paths))
    if DATE_FILTER:
        paths = [p for p in paths if infer_date_from_path(p) == DATE_FILTER]
    return paths


def discover_zip_paths():
    zips = []
    if KAGGLE_INPUT.exists():
        zips += list(KAGGLE_INPUT.glob("**/pokemon-tcg-ai-battle-episodes-*.zip"))
        zips += list(KAGGLE_INPUT.glob("**/*.zip"))
    zips += list((LOCAL_ARTIFACT / "dataset_zips").glob("pokemon-tcg-ai-battle-episodes-*.zip"))
    zips = sorted(set(zips))
    if DATE_FILTER:
        zips = [p for p in zips if DATE_FILTER in str(p)]
    return zips

json_paths = discover_json_paths()
zip_paths = discover_zip_paths()
print("JSONファイル数:", len(json_paths))
print("zipファイル数:", len(zip_paths))
print("先頭JSONパス:", json_paths[:3])
print("先頭zipパス:", zip_paths[:3])




In [ ]:
def zip_json_inventory(zpath):
    try:
        with zipfile.ZipFile(zpath) as zf:
            names = sorted(n for n in zf.namelist() if n.endswith(".json") and Path(n).stem.isdigit())
        return len(names)
    except Exception:
        return np.nan


input_inventory_rows = []
for p in json_paths:
    input_inventory_rows.append({
        "source_type": "json",
        "source_path": str(p),
        "date": infer_date_from_path(p),
        "episode_files": 1,
    })
for p in zip_paths:
    input_inventory_rows.append({
        "source_type": "zip",
        "source_path": str(p),
        "date": infer_date_from_path(p),
        "episode_files": zip_json_inventory(p),
    })
input_inventory = pd.DataFrame(input_inventory_rows)
if len(input_inventory):
    input_inventory["episode_files"] = pd.to_numeric(input_inventory["episode_files"], errors="coerce")
    json_by_date = input_inventory[input_inventory["source_type"] == "json"].groupby("date", dropna=False).agg(
        json_sources=("source_path", "count"),
        json_episode_files=("episode_files", "sum"),
    ).reset_index()
    zip_by_date = input_inventory[input_inventory["source_type"] == "zip"].groupby("date", dropna=False).agg(
        zip_sources=("source_path", "count"),
        zip_episode_files=("episode_files", "sum"),
    ).reset_index()
    input_by_date = json_by_date.merge(zip_by_date, on="date", how="outer")
    for col in ["json_sources", "zip_sources", "json_episode_files", "zip_episode_files"]:
        if col not in input_by_date:
            input_by_date[col] = 0
        input_by_date[col] = pd.to_numeric(input_by_date[col], errors="coerce").fillna(0)
    input_by_date["sources"] = input_by_date["json_sources"] + input_by_date["zip_sources"]
    # If both expanded JSON and zip are attached, they usually contain the same official episodes.
    # Count the expanded JSON side first to avoid presenting duplicate input coverage.
    input_by_date["discovered_episode_files"] = np.where(
        input_by_date["json_episode_files"] > 0,
        input_by_date["json_episode_files"],
        input_by_date["zip_episode_files"],
    )
    input_total_episode_files = int(input_inventory["episode_files"].fillna(0).sum())
    input_total_unique_episode_files = int(input_by_date["discovered_episode_files"].fillna(0).sum())
    print("入力ファイルから数えたリプレイJSON総数:", input_total_episode_files)
    print("重複を除いた推定リプレイJSON数:", input_total_unique_episode_files)
    display_ja(input_by_date)
else:
    input_by_date = pd.DataFrame(columns=["date", "sources", "discovered_episode_files"])
    input_total_episode_files = 0
    input_total_unique_episode_files = 0
    print("入力ファイルからリプレイJSON数を数えられませんでした。")




In [ ]:
def iter_replay_records(max_episodes=MAX_EPISODES):
    count = 0
    seen_episode_ids = set()
    for path in json_paths:
        if max_episodes is not None and count >= max_episodes:
            return
        try:
            data = json.loads(path.read_text(encoding="utf-8"))
            episode_id = (data.get("info") or {}).get("EpisodeId") or data.get("id") or Path(str(path)).stem
            if episode_id in seen_episode_ids:
                continue
            seen_episode_ids.add(episode_id)
            yield path, data
            count += 1
        except Exception as exc:
            print("JSON読み込み失敗", path, repr(exc))

    for zpath in zip_paths:
        if max_episodes is not None and count >= max_episodes:
            return
        try:
            with zipfile.ZipFile(zpath) as zf:
                names = sorted(n for n in zf.namelist() if n.endswith(".json") and Path(n).stem.isdigit())
                for name in names:
                    if max_episodes is not None and count >= max_episodes:
                        return
                    with zf.open(name) as f:
                        data = json.loads(f.read().decode("utf-8"))
                    episode_id = (data.get("info") or {}).get("EpisodeId") or data.get("id") or Path(name).stem
                    if episode_id in seen_episode_ids:
                        continue
                    seen_episode_ids.add(episode_id)
                    pseudo = Path(str(zpath) + "::" + name)
                    yield pseudo, data
                    count += 1
        except Exception as exc:
            print("zip読み込み失敗", zpath, repr(exc))

# 先頭リプレイを軽く確認します。
first = next(iter_replay_records(max_episodes=1), None)
if first:
    p, replay = first
    print("サンプルパス:", p)
    print("キー:", sorted(replay.keys()))
    print("info:", replay.get("info"))
    print("報酬:", replay.get("rewards"), "状態:", replay.get("statuses"), "ステップ数:", len(replay.get("steps", [])))
else:
    print("リプレイJSONが見つかりません。日別の公式上位対戦データセットを少なくとも1つ入力に追加してください。")




## ネストしたリプレイJSONを読むヘルパー

リプレイの細かい形は少し揺れるため、防御的に取り出します。ここで読むものは次の通りです。

- 対戦単位の基本情報
- プレイヤーごとの勝敗行
- 見えている各ステップの選択アクション
- バトル場、ベンチ、トラッシュ、サイド、見える手札などから取れる公開カード名
- 頻出イベントを見るためのログ文字列




In [ ]:
def get_team_names(replay):
    info = replay.get("info") or {}
    teams = info.get("TeamNames") or []
    if len(teams) < 2:
        agents = info.get("Agents") or []
        teams = [a.get("Name", f"player_{i}") for i, a in enumerate(agents)]
    while len(teams) < 2:
        teams.append(f"player_{len(teams)}")
    return teams[:2]


def player_records_from_step(step):
    if isinstance(step, list):
        return step
    if isinstance(step, dict):
        if "players" in step and isinstance(step["players"], list):
            return step["players"]
        return [step]
    return []


def obs_from_record(record):
    if not isinstance(record, dict):
        return {}
    obs = record.get("observation") or {}
    if isinstance(obs, str):
        try:
            obs = json.loads(obs)
        except Exception:
            obs = {}
    return obs if isinstance(obs, dict) else {}


def current_from_obs(obs):
    cur = obs.get("current") or {}
    return cur if isinstance(cur, dict) else {}


def safe_len(x):
    return len(x) if isinstance(x, list) else 0


def iter_dicts(value):
    if isinstance(value, dict):
        yield value
        for v in value.values():
            yield from iter_dicts(v)
    elif isinstance(value, list):
        for v in value:
            yield from iter_dicts(v)


def card_label_from_obj(obj):
    if not isinstance(obj, dict):
        return None
    for key in ["name", "Name", "cardName", "card_name", "label"]:
        v = obj.get(key)
        if isinstance(v, str) and v.strip():
            return v.strip()
    if "id" in obj:
        return f"card_id:{obj.get('id')}"
    return None


def collect_card_labels(value):
    labels = []
    for d in iter_dicts(value):
        label = card_label_from_obj(d)
        if label:
            labels.append(label)
    return labels


def flatten_action(action):
    if action is None:
        return None
    if isinstance(action, (str, int, float, bool)):
        return str(action)
    try:
        return json.dumps(action, ensure_ascii=False, sort_keys=True)[:500]
    except Exception:
        return str(action)[:500]


def action_family(action_text):
    if action_text is None or action_text == "None":
        return "none"
    text = action_text.lower()
    rules = [
        ("attack", ["attack", "ワザ", "damage"]),
        ("play_card", ["play", "card", "use", "trainer", "supporter", "item"]),
        ("attach_energy", ["energy", "attach", "エネルギー"]),
        ("retreat_switch", ["retreat", "switch", "escape", "交代"]),
        ("evolve", ["evolve", "evolution", "進化"]),
        ("bench", ["bench", "ベンチ"]),
        ("choose_select", ["select", "choice", "choose"]),
        ("pass_end", ["pass", "end", "done"]),
    ]
    for family, needles in rules:
        if any(n in text for n in needles):
            return family
    return "other"


def logs_from_obs(obs):
    logs = obs.get("logs")
    if logs is None:
        return []
    if isinstance(logs, list):
        return [str(x) for x in logs]
    return [str(logs)]




## 分析用テーブルを作成

ここでは主に4つのテーブルを作ります。

- 対戦テーブル: リプレイごとに1行
- プレイヤーテーブル: リプレイごとに2行
- アクションテーブル: アクションが出たプレイヤー記録ごとに1行
- カード観測テーブル: 観測から拾えた公開/可視カードを、対戦/プレイヤー/カード単位に集約した行

全件スキャンではカード観測が非常に多くなります。
そのため、カードは生の全ステップ行ではなく `observations` に観測回数を集約して保存します。




In [ ]:
episode_rows = []
player_rows = []
action_rows = []
card_rows = []
log_counter = collections.Counter()

for path, replay in iter_replay_records(MAX_EPISODES):
    episode_id = (replay.get("info") or {}).get("EpisodeId") or replay.get("id") or Path(str(path)).stem
    try:
        episode_id_int = int(episode_id)
    except Exception:
        episode_id_int = None
    date = infer_date_from_path(path)
    teams = get_team_names(replay)
    rewards = replay.get("rewards") or [None, None]
    statuses = replay.get("statuses") or [None, None]
    steps = replay.get("steps") or []
    config = replay.get("configuration") or {}
    seed = config.get("seed")

    episode_rows.append({
        "episode_id": episode_id_int,
        "date": date,
        "path": str(path),
        "source_type": "zip" if "::" in str(path) else "json",
        "steps": len(steps),
        "seed": seed,
        "team_0": teams[0],
        "team_1": teams[1],
        "reward_0": rewards[0] if len(rewards) > 0 else None,
        "reward_1": rewards[1] if len(rewards) > 1 else None,
        "status_0": statuses[0] if len(statuses) > 0 else None,
        "status_1": statuses[1] if len(statuses) > 1 else None,
    })

    for i in range(2):
        reward = rewards[i] if i < len(rewards) else None
        player_rows.append({
            "episode_id": episode_id_int,
            "date": date,
            "player_index": i,
            "team": teams[i],
            "reward": reward,
            "is_win": reward == 1,
            "is_loss": reward == -1,
            "status": statuses[i] if i < len(statuses) else None,
            "opponent": teams[1 - i],
            "episode_steps": len(steps),
        })

    episode_card_counter = collections.Counter()
    for step_idx, step in enumerate(steps):
        for player_idx, record in enumerate(player_records_from_step(step)):
            if not isinstance(record, dict):
                continue
            obs = obs_from_record(record)
            cur = current_from_obs(obs)
            action_text = flatten_action(record.get("action"))
            action_ids = parse_action_ids(action_text)
            selected = record.get("selected", obs.get("selected"))
            select_obj = obs.get("select")
            select_info = select_summary(select_obj)
            reward = record.get("reward")
            status = record.get("status")
            if action_text is not None and action_ids:
                resolved_action = resolved_action_display(action_text, select_obj, cur)
                board_info = visible_board_summary(cur, player_idx)
                row = {
                    "episode_id": episode_id_int,
                    "date": date,
                    "step_idx": step_idx,
                    "player_index": player_idx,
                    "team": teams[player_idx] if player_idx < len(teams) else f"player_{player_idx}",
                    "is_winner_player": (rewards[player_idx] == 1) if player_idx < len(rewards) else None,
                    "action": action_text,
                    "action_ids": ", ".join(map(str, action_ids)),
                    "action_cards_ja": resolved_action,
                    "resolved_action_ja": resolved_action,
                    "action_kind_ja": action_kind_from_resolved(resolved_action),
                    "action_family": action_family(action_text),
                    "selected": flatten_action(selected),
                    "turn": cur.get("turn"),
                    "first_player": cur.get("firstPlayer"),
                    "your_index": cur.get("yourIndex"),
                    "reward_at_step": reward,
                    "status_at_step": status,
                    "obs_step": obs.get("step"),
                    "remaining_overage_time": obs.get("remainingOverageTime"),
                }
                row.update(select_info)
                row.update(board_info)
                action_rows.append(row)

            for line in logs_from_obs(obs):
                if line:
                    log_counter[line[:240]] += 1

            # Keep card extraction broad but public/visible only. Hidden hands may be null or absent.
            for name in collect_card_labels(cur):
                team_name = teams[player_idx] if player_idx < len(teams) else f"player_{player_idx}"
                is_winner = (rewards[player_idx] == 1) if player_idx < len(rewards) else None
                episode_card_counter[(player_idx, team_name, is_winner, name)] += 1

    for (player_idx, team_name, is_winner, name), observations in episode_card_counter.items():
        card_rows.append({
            "episode_id": episode_id_int,
            "date": date,
            "player_index": player_idx,
            "team": team_name,
            "is_winner_player": is_winner,
            "card_name": name,
            "observations": observations,
        })

episodes_df = pd.DataFrame(episode_rows)
players_df = pd.DataFrame(player_rows)
actions_df = pd.DataFrame(action_rows)
cards_df = pd.DataFrame(card_rows)
if len(cards_df):
    cards_df["winner_observation_count"] = np.where(cards_df["is_winner_player"] == True, cards_df["observations"], 0)

print("対戦テーブル形状", episodes_df.shape)
print("プレイヤーテーブル形状", players_df.shape)
print("アクションテーブル形状", actions_df.shape)
print("カード観測テーブル形状", cards_df.shape)




In [ ]:
display_ja(episodes_df, head=5)
display_ja(players_df, head=5)
display_ja(actions_df, columns=[
    "episode_id", "date", "step_idx", "team", "is_winner_player",
    "select_context_ja", "select_type_ja", "select_option_count", "select_summary_ja",
    "effect_card_ja", "context_card_ja", "resolved_action_ja", "action_ids",
], head=8)
display_ja(cards_df, head=5)




## 日別カバレッジとスコア状況




In [ ]:
if len(manifest):
    daily_scan = episodes_df.groupby("date", dropna=False).agg(
        scanned_episodes=("episode_id", "count"),
        mean_steps=("steps", "mean"),
        median_steps=("steps", "median"),
    ).reset_index()
    coverage = manifest.merge(daily_scan, on="date", how="left") if "date" in manifest else daily_scan
    display_ja(coverage)
else:
    display_ja(episodes_df.groupby("date", dropna=False).agg(scanned_episodes=("episode_id", "count"), mean_steps=("steps", "mean")).reset_index())




In [ ]:
if HAS_PLOT and len(manifest):
    fig, ax1 = plt.subplots(figsize=(12.5, 4.8))
    x = np.arange(len(manifest))
    bars = ax1.bar(x, manifest["episode_count"].fillna(0), color=PALETTE["blue"], alpha=0.82, label="対戦数")
    ax1.set_ylabel("対戦数")
    ax1.set_xticks(x)
    ax1.set_xticklabels(manifest["date"], rotation=30, ha="right")
    ax1.bar_label(bars, labels=[f"{int(v):,}" if pd.notna(v) else "" for v in manifest["episode_count"]], padding=3, fontsize=8, color="#364152")
    ax2 = ax1.twinx()
    ax2.plot(x, manifest["top_avg_score"], color=PALETTE["orange"], marker="o", linewidth=2.5, label="上位平均スコア")
    ax2.plot(x, manifest["median_avg_score"], color=PALETTE["green"], marker="o", linewidth=2.5, label="中央値平均スコア")
    ax2.set_ylabel("平均スコア")
    polish_axes(ax1)
    ax2.spines["top"].set_visible(False)
    lines, labels = ax1.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax1.legend(lines + lines2, labels + labels2, frameon=False, loc="upper left")
    finish_plot("01_manifest_episode_count_and_scores.png", "公式上位対戦の日別件数とスコア", "Kaggle公式manifestに載っている日別データ量とスコア水準")




## このNotebookはデータのどこまでを読んだか

`MAX_EPISODES` はNotebookを軽く回すための上限です。
そのため、ここで「manifestに何件あるか」「入力ファイルから何件見つかったか」「今回のEDAで何件読んだか」を分けて確認します。
読込率が低い場合、この後の勝率・カード・行動分析は「全体」ではなく「先頭から読んだサンプル」の分析として見ます。




In [ ]:
if len(episodes_df):
    read_by_date = episodes_df.groupby("date", dropna=False).agg(read_episodes=("episode_id", "nunique")).reset_index()
else:
    read_by_date = pd.DataFrame(columns=["date", "read_episodes"])

coverage_parts = []
if len(manifest):
    coverage_parts.append(manifest[["date", "episode_count"]].rename(columns={"episode_count": "manifest_episodes"}))
if "input_by_date" in globals() and len(input_by_date):
    coverage_parts.append(input_by_date[["date", "discovered_episode_files"]])
coverage_parts.append(read_by_date)
coverage_by_date = coverage_parts[0]
for part in coverage_parts[1:]:
    coverage_by_date = coverage_by_date.merge(part, on="date", how="outer")

for col in ["manifest_episodes", "discovered_episode_files", "read_episodes"]:
    if col not in coverage_by_date:
        coverage_by_date[col] = np.nan
    coverage_by_date[col] = pd.to_numeric(coverage_by_date[col], errors="coerce")
coverage_by_date["read_vs_manifest_rate"] = coverage_by_date["read_episodes"] / coverage_by_date["manifest_episodes"]
coverage_by_date["read_vs_discovered_rate"] = coverage_by_date["read_episodes"] / coverage_by_date["discovered_episode_files"]
coverage_by_date = coverage_by_date.sort_values("date", na_position="last")
display_ja(coverage_by_date)

manifest_total_episodes = int(manifest["episode_count"].dropna().sum()) if len(manifest) and "episode_count" in manifest else np.nan
discovered_total_episodes = int(input_total_unique_episode_files) if "input_total_unique_episode_files" in globals() else np.nan
read_total_episodes = int(episodes_df["episode_id"].nunique()) if len(episodes_df) else 0
coverage_totals = pd.DataFrame([
    {"scope": "manifest上の公式件数", "episodes": manifest_total_episodes},
    {"scope": "入力ファイルから数えた件数", "episodes": discovered_total_episodes},
    {"scope": "今回EDAで読んだ件数", "episodes": read_total_episodes},
])
coverage_totals["read_rate_vs_this_scope"] = coverage_totals["episodes"].map(lambda x: read_total_episodes / x if pd.notna(x) and x else np.nan)
display_ja(coverage_totals)




In [ ]:
if HAS_PLOT and len(coverage_by_date):
    plot_cov = coverage_by_date.copy()
    plot_cov["date_label"] = plot_cov["date"].fillna("日付不明").astype(str)
    long_cov = plot_cov.melt(
        id_vars="date_label",
        value_vars=["manifest_episodes", "discovered_episode_files", "read_episodes"],
        var_name="種類",
        value_name="対戦数",
    )
    long_cov["種類"] = long_cov["種類"].map({
        "manifest_episodes": "manifest上の件数",
        "discovered_episode_files": "入力ファイル内の件数",
        "read_episodes": "今回読んだ件数",
    })
    fig, ax = plt.subplots(figsize=(12.5, 5.2))
    sns.barplot(data=long_cov, x="date_label", y="対戦数", hue="種類", ax=ax, palette=[PALETTE["gray"], PALETTE["blue"], PALETTE["green"]])
    ax.set_xlabel("日付")
    ax.set_ylabel("対戦数")
    ax.tick_params(axis="x", rotation=30)
    ax.legend(frameon=False, loc="upper left")
    finish_plot("21_dataset_coverage_by_date.png", "日別にどこまで読んだか", "manifest、入力ファイル、今回EDAで読んだ件数を分けて確認")




In [ ]:
if HAS_PLOT and len(coverage_totals):
    plot_total = coverage_totals.dropna(subset=["episodes"]).copy()
    plot_total = plot_total.sort_values("episodes", ascending=True)
    fig, ax = plt.subplots(figsize=(10.2, 4.8))
    colors = [PALETTE["green"] if s == "今回EDAで読んだ件数" else PALETTE["blue"] for s in plot_total["scope"]]
    ax.barh(plot_total["scope"], plot_total["episodes"], color=colors, alpha=0.84)
    for _, row in plot_total.iterrows():
        ax.text(row["episodes"], row["scope"], f" {int(row['episodes']):,}", va="center", fontsize=9, color="#334155")
    ax.set_xlabel("対戦数")
    ax.set_ylabel("")
    subtitle = f"今回の読込率: manifest比 {read_total_episodes / manifest_total_episodes:.1%}" if pd.notna(manifest_total_episodes) and manifest_total_episodes else "今回読んだ件数と入力母数を比較"
    finish_plot("22_dataset_read_coverage_bar.png", "今回のEDAは全体のどれくらいか", subtitle)




## 対戦の長さと勝敗分布

長い試合はリソース循環、山札切れリスク、展開の遅さを見る手がかりになります。
短すぎる試合は序盤方針の確認に向いています。




In [ ]:
if len(episodes_df):
    display(episodes_df["steps"].describe().to_frame().T)
    display(players_df.groupby(["reward", "status"], dropna=False).size().reset_index(name="rows"))




In [ ]:
if HAS_PLOT and len(episodes_df):
    fig, ax = plt.subplots(figsize=(11, 4.8))
    sns.histplot(data=episodes_df, x="steps", bins=42, kde=True, ax=ax, color=PALETTE["blue"], edgecolor="white", alpha=0.82)
    median_steps = episodes_df["steps"].median()
    p90_steps = episodes_df["steps"].quantile(0.90)
    ax.axvline(median_steps, color=PALETTE["orange"], linestyle="--", linewidth=2, label=f"中央値={median_steps:.0f}")
    ax.axvline(p90_steps, color=PALETTE["red"], linestyle=":", linewidth=2, label=f"p90={p90_steps:.0f}")
    ax.set_xlabel("リプレイ内ステップ数")
    ax.set_ylabel("対戦数")
    ax.legend(frameon=False)
    finish_plot("02_replay_length_distribution.png", "対戦の長さ分布", "短い対戦は序盤方針、長い対戦はリソース循環の確認候補")




## チーム別・対面別の集計

公式上位対戦でよく出るチームと、見直す価値がありそうな対面を確認します。




In [ ]:
if len(players_df):
    team_summary = players_df.groupby("team").agg(
        games=("player_index", "size"),
        wins=("is_win", "sum"),
        losses=("is_loss", "sum"),
        mean_reward=("reward", "mean"),
        mean_episode_steps=("episode_steps", "mean"),
    ).reset_index()
    team_summary["win_rate"] = team_summary["wins"] / team_summary["games"]
    team_summary = team_summary.sort_values(["wins", "games", "mean_reward"], ascending=False)
    display_ja(team_summary, head=50)

    if HAS_PLOT:
        plot_teams = team_summary.head(25).sort_values("wins").copy()
        plot_teams["team_label"] = plot_teams["team"].map(safe_plot_label)
        fig, ax = plt.subplots(figsize=(10.5, max(5, len(plot_teams) * 0.30)))
        ax.barh(plot_teams["team_label"], plot_teams["wins"], color=PALETTE["green"], alpha=0.88, label="勝ち")
        ax.barh(plot_teams["team_label"], plot_teams["losses"], left=plot_teams["wins"], color=PALETTE["red"], alpha=0.45, label="負け")
        for _, row in plot_teams.iterrows():
            ax.text(row["wins"] + row["losses"] + 0.15, row["team_label"], f"{row['win_rate']:.0%}", va="center", fontsize=8, color="#46505c")
        ax.set_xlabel("公式上位対戦内のプレイヤー行数")
        ax.legend(frameon=False, loc="lower right")
        finish_plot("03_top_teams_win_loss_bars.png", "公式上位対戦内で多く見えたチーム", "棒の長さは観測試合数、右端はスキャン範囲内の勝率")




In [ ]:
if len(players_df):
    matchup = players_df.groupby(["team", "opponent"]).agg(
        games=("player_index", "size"),
        wins=("is_win", "sum"),
        mean_reward=("reward", "mean"),
    ).reset_index()
    matchup["win_rate"] = matchup["wins"] / matchup["games"]
    matchup = matchup[matchup["games"] >= 1].sort_values(["games", "win_rate"], ascending=False)
    display_ja(matchup, head=60)

    if HAS_PLOT and len(matchup):
        top_pairs = matchup.head(25).copy()
        top_pairs["pair"] = top_pairs["team"].map(safe_plot_label) + " 対 " + top_pairs["opponent"].map(safe_plot_label)
        top_pairs = top_pairs.sort_values("games")
        fig, ax = plt.subplots(figsize=(11.5, max(5, len(top_pairs) * 0.32)))
        colors = [PALETTE["green"] if v >= 0.5 else PALETTE["red"] for v in top_pairs["win_rate"]]
        ax.barh(top_pairs["pair"], top_pairs["games"], color=colors, alpha=0.80)
        for _, row in top_pairs.iterrows():
            ax.text(row["games"] + 0.05, row["pair"], f"{row['win_rate']:.0%}", va="center", fontsize=8, color="#46505c")
        ax.set_xlabel("観測されたプレイヤー行数")
        finish_plot("04_frequent_matchups.png", "よく出る対面", "緑は表示側チームの勝率がスキャン範囲内で50%以上")




## 対戦時間と勝率の関係

1試合の長さは勝者と敗者で同じなので、単純に「短い試合の勝率」を出すと必ず50%付近になります。
そこでチーム単位に集計して、平均対戦時間が短い/長いチームほど勝率がどう変わるかを見ます。
あわせて、各チームが「勝つ時は短く決めるのか」「長引いた時に勝ちやすいのか」も確認します。
公式Top episodeはチームごとの試合数が少ないため、線の傾きは断定材料ではなく、詳しく見るチームを選ぶための目安です。




In [ ]:
if len(players_df):
    team_time = players_df.groupby("team").agg(
        games=("player_index", "size"),
        wins=("is_win", "sum"),
        win_rate=("is_win", "mean"),
        avg_steps=("episode_steps", "mean"),
        median_steps=("episode_steps", "median"),
    ).reset_index()
    win_steps = players_df[players_df["is_win"]].groupby("team")["episode_steps"].mean().rename("win_avg_steps")
    loss_steps = players_df[players_df["is_loss"]].groupby("team")["episode_steps"].mean().rename("loss_avg_steps")
    team_time = team_time.merge(win_steps, on="team", how="left").merge(loss_steps, on="team", how="left")
    team_time["win_minus_loss_steps"] = team_time["win_avg_steps"] - team_time["loss_avg_steps"]
    team_time = team_time.sort_values(["games", "win_rate"], ascending=False)
    display_ja(team_time, head=60)

    min_games_for_time = max(3, int(team_time["games"].quantile(0.35))) if len(team_time) else 3
    team_time_plot = team_time[team_time["games"] >= min_games_for_time].copy()
    if len(team_time_plot) >= 3:
        team_time_plot["team_label"] = team_time_plot["team"].map(lambda x: safe_plot_label(x, 18))
        team_time_plot["勝率"] = team_time_plot["win_rate"]
        team_time_plot["平均ステップ数"] = team_time_plot["avg_steps"]
        team_time_plot["試合数"] = team_time_plot["games"]

        fig, ax = plt.subplots(figsize=(10.5, 6.0))
        sns.scatterplot(
            data=team_time_plot,
            x="平均ステップ数",
            y="勝率",
            size="試合数",
            hue="試合数",
            palette="viridis",
            sizes=(50, 320),
            alpha=0.78,
            ax=ax,
            legend="brief",
        )
        if len(team_time_plot) >= 5:
            sns.regplot(data=team_time_plot, x="平均ステップ数", y="勝率", scatter=False, ax=ax, color="#334155", line_kws={"linewidth": 1.4, "alpha": 0.6})
        for _, row in team_time_plot.sort_values("試合数", ascending=False).head(18).iterrows():
            ax.text(row["平均ステップ数"] + 0.8, row["勝率"], row["team_label"], fontsize=8, color="#334155")
        ax.axhline(0.5, color="#64748b", linestyle="--", linewidth=1)
        ax.set_ylim(max(0, team_time_plot["勝率"].min() - 0.08), min(1, team_time_plot["勝率"].max() + 0.08))
        ax.set_xlabel("平均対戦ステップ数")
        ax.set_ylabel("勝率")
        finish_plot("18_team_winrate_vs_duration.png", "平均対戦時間と勝率", "右ほど長期戦寄り、上ほど勝率が高い。点の大きさは試合数")

        diff_plot = team_time_plot.dropna(subset=["win_minus_loss_steps"]).copy()
        diff_plot = diff_plot.reindex(diff_plot["win_minus_loss_steps"].abs().sort_values(ascending=False).index).head(25)
        if len(diff_plot):
            diff_plot = diff_plot.sort_values("win_minus_loss_steps")
            colors = [PALETTE["green"] if v < 0 else PALETTE["orange"] for v in diff_plot["win_minus_loss_steps"]]
            fig, ax = plt.subplots(figsize=(10.5, max(5, len(diff_plot) * 0.32)))
            ax.barh(diff_plot["team"].map(lambda x: safe_plot_label(x, 28)), diff_plot["win_minus_loss_steps"], color=colors, alpha=0.84)
            ax.axvline(0, color="#334155", linewidth=1)
            ax.set_xlabel("勝った試合の平均ステップ数 - 負けた試合の平均ステップ数")
            ax.set_ylabel("")
            finish_plot("19_team_win_loss_duration_delta.png", "勝つ時の長さはチームで違う", "左は勝つ時に短く、右は勝つ時に長い傾向")




## アクションEDA

アクション値をそのまま見ると、候補番号やカードIDが混ざって読みにくくなります。

注意: アクション値には、カードIDと「その場面の候補リスト番号」が混ざります。
このノートブックでは候補番号をそのまま集計せず、`select.option` の中身を引き直して、
可能な限り「どのカード/ワザ/終了操作を実際に選んだか」まで復元してから集計します。
まず大分類、次に具体的なカード名/ワザ/終了操作を見ます。




In [ ]:
if len(actions_df):
    action_kind_summary = actions_df.groupby(["action_kind_ja", "is_winner_player"], dropna=False).size().reset_index(name="count")
    display_ja(action_kind_summary.sort_values("count", ascending=False))

    action_top = actions_df.groupby([
        "action_kind_ja", "select_context_ja", "select_type_ja", "resolved_action_ja"
    ], dropna=False).size().reset_index(name="count").sort_values("count", ascending=False)
    display_ja(action_top, columns=[
        "action_kind_ja", "select_context_ja", "select_type_ja", "resolved_action_ja", "count"
    ], head=80)




In [ ]:
if HAS_PLOT and len(actions_df):
    fam = actions_df["action_kind_ja"].value_counts().head(14).sort_values()
    fig, ax = plt.subplots(figsize=(10.5, 4.5))
    sns.barplot(x=fam.values, y=fam.index, ax=ax, color=PALETTE["blue"], orient="h")
    ax.set_xlabel("レコード数")
    ax.set_ylabel("")
    finish_plot("05_action_family_frequency.png", "復元した行動カテゴリの頻度", "候補番号ではなく、実際に選んだカード/ワザ/終了操作から分類")




In [ ]:
if len(actions_df):
    turn_action = actions_df.copy()
    turn_action["turn_bin"] = pd.cut(pd.to_numeric(turn_action["turn"], errors="coerce"), bins=[-1, 1, 3, 6, 10, 20, 999], labels=["0-1", "2-3", "4-6", "7-10", "11-20", "21+"])
    display_ja(turn_action.groupby(["turn_bin", "action_kind_ja"], observed=False).size().reset_index(name="count").sort_values(["turn_bin", "count"], ascending=[True, False]), head=80)




In [ ]:
if len(actions_df):
    action_id_counter = collections.Counter()
    winner_action_id_counter = collections.Counter()
    loser_action_id_counter = collections.Counter()
    action_label_counter = collections.Counter()
    winner_action_label_counter = collections.Counter()
    loser_action_label_counter = collections.Counter()
    for row in actions_df[["action", "resolved_action_ja", "is_winner_player"]].itertuples(index=False):
        ids = parse_action_ids(row.action)
        action_id_counter.update(ids)
        labels = [label.strip() for label in str(row.resolved_action_ja).split(" / ") if label.strip() and label.strip().lower() != "nan"]
        action_label_counter.update(labels)
        if row.is_winner_player is True:
            winner_action_id_counter.update(ids)
            winner_action_label_counter.update(labels)
        elif row.is_winner_player is False:
            loser_action_id_counter.update(ids)
            loser_action_label_counter.update(labels)
    action_ids_df = pd.DataFrame([
        {"action_id": k, "count": v, "winner_count": winner_action_id_counter[k], "loser_count": loser_action_id_counter[k]}
        for k, v in action_id_counter.most_common(200)
    ])
    if len(action_ids_df):
        action_ids_df["winner_share"] = action_ids_df["winner_count"] / action_ids_df["count"]
        action_ids_df["action_label_ja"] = action_ids_df["action_id"].map(lambda x: f"生値 {x}")
    action_values_df = pd.DataFrame([
        {
            "action_value_label_ja": k,
            "count": v,
            "winner_count": winner_action_label_counter[k],
            "loser_count": loser_action_label_counter[k],
        }
        for k, v in action_label_counter.most_common(200)
    ])
    if len(action_values_df):
        action_values_df["winner_share"] = action_values_df["winner_count"] / action_values_df["count"]
    display_ja(action_values_df, head=80)

    if HAS_PLOT and len(action_values_df):
        plot_ids = action_values_df.head(30).sort_values("count")
        fig, ax = plt.subplots(figsize=(10.5, 7.5))
        sns.barplot(data=plot_ids, x="count", y="action_value_label_ja", ax=ax, color=PALETTE["pink"], orient="h")
        ax.set_xlabel("復元した行動内容の出現頻度")
        ax.set_ylabel("復元した行動内容")
        finish_plot("06_top_action_ids.png", "よく選ばれたアクション内容", "候補番号はselect.optionを引き直してカード/ワザ/終了操作に復元")




In [ ]:
if HAS_PLOT and len(actions_df):
    heat = turn_action.groupby(["turn_bin", "action_kind_ja"], observed=False).size().unstack(fill_value=0)
    if len(heat) and heat.to_numpy().sum() > 0:
        top_cols = heat.sum().sort_values(ascending=False).head(8).index
        heat = heat.loc[:, top_cols]
        fig, ax = plt.subplots(figsize=(11.5, 4.8))
        sns.heatmap(heat, annot=True, fmt="d", cmap="Blues", linewidths=0.5, linecolor="white", ax=ax)
        ax.set_xlabel("行動カテゴリ")
        ax.set_ylabel("ターン帯")
        finish_plot("11_turn_bin_action_heatmap.png", "ターン帯別に何をしているか", "序盤・中盤・終盤で増えるカード使用/ワザ/番終了を確認")




## 盤面条件ごとに、実際に選ばれた手を見る

「選択肢0が多い」のような候補番号は、戦略としてはほぼ意味がありません。
ここでは、ターン帯、自分/相手のバトル場、自分ベンチ数で盤面を粗くまとめ、
その盤面条件でどのカード・ワザ・終了操作がよく選ばれたかを見ます。

これはそのまま強いAIではありませんが、次の2つには使えます。

- 単純な相関/頻度ベースの方策: 似た盤面で一番よく選ばれた合法手を優先する
- 教師あり模倣学習の入口: デッキ構成、公開盤面、合法手候補を入力にして、観測された選択手を当てる

ただし、候補手が合法かどうかはその局面ごとに違うため、実装では「全行動から直接1つ選ぶ」のではなく、
その場の合法手リストを特徴化して順位付けする形にするのが現実的です。




In [ ]:
if len(actions_df):
    strategy_base = actions_df[
        actions_df["resolved_action_ja"].notna()
        & actions_df["board_key_ja"].notna()
        & ~actions_df["action_kind_ja"].isin(["補助値", "不明"])
    ].copy()
    board_action_counts = (
        strategy_base
        .groupby([
            "board_key_ja", "own_active_ja", "opp_active_ja", "own_bench_count",
            "action_kind_ja", "resolved_action_ja"
        ], dropna=False)
        .agg(
            count=("resolved_action_ja", "size"),
            winner_count=("is_winner_player", "sum"),
            episodes=("episode_id", "nunique"),
        )
        .reset_index()
    )
    board_totals = board_action_counts.groupby("board_key_ja")["count"].transform("sum")
    board_action_counts["selected_share"] = board_action_counts["count"] / board_totals
    board_action_counts["winner_share"] = board_action_counts["winner_count"] / board_action_counts["count"].replace(0, np.nan)
    board_action_counts["confidence_note"] = np.select(
        [
            board_action_counts["count"] >= 50,
            board_action_counts["count"] >= 15,
        ],
        [
            "観測多め。方策候補として優先確認",
            "中程度。リプレイで確認",
        ],
        default="観測少なめ。断定しない",
    )
    board_action_counts = board_action_counts.sort_values(
        ["count", "selected_share", "winner_share"], ascending=False
    )
    display_ja(board_action_counts, columns=[
        "board_key_ja", "action_kind_ja", "resolved_action_ja", "count",
        "selected_share", "winner_share", "episodes", "confidence_note",
    ], head=120)




In [ ]:
if HAS_PLOT and "board_action_counts" in globals() and len(board_action_counts):
    plot_board = board_action_counts.copy()
    plot_board = plot_board[plot_board["count"] >= max(MIN_ACTION_COUNT, 20)].copy()
    plot_board["board_pair_ja"] = (
        plot_board["board_key_ja"].map(lambda x: safe_plot_label(x, 40))
        + " → "
        + plot_board["resolved_action_ja"].map(lambda x: safe_plot_label(x, 34))
    )
    plot_pairs = plot_board.sort_values(["count", "selected_share"], ascending=False).head(20).sort_values("count")
    if len(plot_pairs):
        fig, ax = plt.subplots(figsize=(12.8, max(7.2, len(plot_pairs) * 0.36)))
        colors = plt.cm.viridis(np.clip(plot_pairs["winner_share"].fillna(0.5).to_numpy(), 0, 1))
        ax.barh(plot_pairs["board_pair_ja"], plot_pairs["count"], color=colors, alpha=0.86)
        sm = plt.cm.ScalarMappable(cmap="viridis", norm=plt.Normalize(0, 1))
        sm.set_array([])
        cbar = fig.colorbar(sm, ax=ax, pad=0.015)
        cbar.set_label("勝者側比率")
        ax.set_xlabel("選択回数")
        ax.set_ylabel("盤面条件 → 復元した選択内容")
        ax.margins(x=0.06)
        finish_plot("23_board_state_to_action_heatmap.png", "盤面条件ごとによく選ばれた手", "横棒は選択回数、色は勝者側比率。候補番号は使っていません")




In [ ]:
if len(actions_df):
    action_signal = actions_df[
        actions_df["resolved_action_ja"].notna()
        & ~actions_df["action_kind_ja"].isin(["補助値", "不明"])
    ].copy()
    action_signal["action_short"] = action_signal["resolved_action_ja"].map(lambda x: safe_plot_label(x, 80))
    corr_rows = []
    for action_name, grp in action_signal.groupby("action_short"):
        if len(grp) < MIN_ACTION_COUNT:
            continue
        corr_rows.append({
            "resolved_action_ja": action_name,
            "count": len(grp),
            "winner_share": float(grp["is_winner_player"].mean()),
            "mean_turn": float(pd.to_numeric(grp["turn"], errors="coerce").mean()),
            "mean_own_bench": float(pd.to_numeric(grp["own_bench_count"], errors="coerce").mean()),
            "mean_own_hand": float(pd.to_numeric(grp["own_hand_count"], errors="coerce").mean()),
        })
    action_correlation = pd.DataFrame(corr_rows)
    if len(action_correlation):
        action_correlation["winrate_lift_vs_all_actions"] = action_correlation["winner_share"] - float(action_signal["is_winner_player"].mean())
        action_correlation = action_correlation.sort_values(["winrate_lift_vs_all_actions", "count"], ascending=False)
        display_ja(action_correlation, head=80)

        if HAS_PLOT:
            plot_corr = pd.concat([
                action_correlation.head(14),
                action_correlation.tail(14),
            ]).drop_duplicates().sort_values("winrate_lift_vs_all_actions")
            fig, ax = plt.subplots(figsize=(11.5, max(5.8, len(plot_corr) * 0.30)))
            colors = [PALETTE["green"] if v >= 0 else PALETTE["red"] for v in plot_corr["winrate_lift_vs_all_actions"]]
            ax.barh(
                plot_corr["resolved_action_ja"].map(lambda x: safe_plot_label(x, 44)),
                plot_corr["winrate_lift_vs_all_actions"],
                color=colors,
                alpha=0.84,
            )
            ax.axvline(0, color="#334155", linewidth=1)
            ax.set_xlabel("全アクション平均に対する勝者側比率の差")
            ax.set_ylabel("復元した選択内容")
            finish_plot("24_action_winrate_lift.png", "勝者側に寄りやすい行動内容", "相関であり因果ではありません。合法手候補の順位付け特徴として使う候補")




## 見えているカードIDの日本語EDA

公開観測オブジェクトからカードIDを拾い、日本語カード名に変換します。
アーキタイプの気配を見るには便利ですが、非公開ゾーンが完全には見えないため、完全なデッキリスト抽出ではありません。




In [ ]:
if len(cards_df):
    card_counts = cards_df.groupby("card_name").agg(
        observations=("observations", "sum"),
        episodes=("episode_id", "nunique"),
        winner_observations=("winner_observation_count", "sum"),
    ).reset_index()
    card_counts["card_id"] = card_counts["card_name"].map(parse_card_id).astype("Int64")
    card_counts["winner_obs_share"] = card_counts["winner_observations"] / card_counts["observations"]
    card_counts = card_counts.sort_values(["episodes", "observations"], ascending=False)
    if len(card_master):
        card_counts = card_counts.merge(card_master, on="card_id", how="left")
    card_counts["display_name_ja"] = card_counts.apply(
        lambda row: row.get("name_ja") if pd.notna(row.get("name_ja")) else card_display_name(row["card_name"], 80),
        axis=1,
    )
    image_paths = extract_official_card_images(card_counts["card_id"].dropna().astype(int).head(36).tolist())
    card_counts["official_image_path"] = card_counts["card_id"].map(lambda x: image_paths.get(int(x)) if pd.notna(x) else None)
    display_cols = [
        "card_id", "display_name_ja", "name_en", "expansion_ja", "collection_no_ja",
        "observations", "episodes", "winner_obs_share", "official_image_path", "card_name",
    ]
    display_ja(card_counts, columns=display_cols, head=80)
else:
    print("見えているカード名/カードIDを抽出できませんでした。")




In [ ]:
if HAS_PLOT and len(cards_df):
    plot_cards = card_counts.head(TOP_N).sort_values("episodes")
    plot_cards = plot_cards.assign(card_label=plot_cards["display_name_ja"].fillna(plot_cards["card_name"]).map(lambda x: safe_plot_label(x, 34)))
    fig, ax = plt.subplots(figsize=(10.5, max(4, TOP_N * 0.26)))
    sns.barplot(data=plot_cards, x="episodes", y="card_label", ax=ax, color=PALETTE["green"], orient="h")
    ax.set_xlabel("見えた対戦数")
    ax.set_ylabel("")
    finish_plot("07_visible_cards_by_episode_count.png", "よく見えるカード（日本語名）", "公開観測のみ: 非公開の手札/山札は完全には見えていません")




In [ ]:
def render_card_gallery(df, n=24):
    if not len(df):
        return
    rows = []
    for row in df.head(n).itertuples(index=False):
        card_id = getattr(row, "card_id", "")
        name_ja = getattr(row, "display_name_ja", None) or getattr(row, "card_name", "")
        name_en = getattr(row, "name_en", "") if hasattr(row, "name_en") else ""
        img_path = getattr(row, "official_image_path", None)
        img = html_img(img_path, 118) if img_path else '<div style="width:118px;height:165px;border-radius:8px;background:#eef2f7;display:flex;align-items:center;justify-content:center;color:#64748b;font-size:12px;text-align:center;">公式画像<br>未抽出</div>'
        rows.append(f"""
        <div style="display:flex;gap:12px;align-items:flex-start;border:1px solid #dbe3ee;border-radius:8px;padding:10px;background:#ffffff;">
          <div>{img}</div>
          <div style="min-width:0;">
            <div style="font-weight:700;color:#0f172a;font-size:14px;">{escape(str(name_ja))}</div>
            <div style="color:#64748b;font-size:12px;margin-top:2px;">ID {escape(str(card_id))} / {escape(str(name_en))}</div>
            <div style="margin-top:8px;color:#334155;font-size:12px;">観測 {getattr(row, "observations", 0):,} / 対戦 {getattr(row, "episodes", 0):,}</div>
            <div style="color:#334155;font-size:12px;">勝者側観測率 {getattr(row, "winner_obs_share", 0):.1%}</div>
          </div>
        </div>
        """)
    html = f"""
    <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(280px,1fr));gap:12px;">
      {''.join(rows)}
    </div>
    """
    try:
        from IPython.display import HTML, display as ipy_display
        ipy_display(HTML(html))
    except Exception:
        print("カードギャラリーはNotebookとして実行すると表示されます。")


if len(cards_df) and "card_counts" in globals():
    render_card_gallery(card_counts, n=24)




## 勝率が高いチームは何を使っているか

ここでは、公式Top episode内で勝率が高いチームを選び、そのチームでよく見えるカードを確認します。
カードは公開観測から拾える範囲なので、完全なデッキリストではありません。
ただし、勝っているチームの盤面・トラッシュ・公開領域に何が出ているかを、カード画像付きでざっと見る用途には向いています。




In [ ]:
if len(cards_df) and "team_summary" in globals() and len(team_summary):
    min_games_for_strong_team = max(3, int(team_summary["games"].quantile(0.50))) if len(team_summary) else 3
    strong_teams = (
        team_summary[team_summary["games"] >= min_games_for_strong_team]
        .sort_values(["win_rate", "wins", "games"], ascending=False)
        .head(6)
        .copy()
    )
    if strong_teams.empty:
        strong_teams = team_summary.sort_values(["win_rate", "wins", "games"], ascending=False).head(6).copy()

    team_card_counts = (
        cards_df[cards_df["team"].isin(strong_teams["team"])]
        .groupby(["team", "card_name"], dropna=False)
        .agg(
            observations=("observations", "sum"),
            episodes=("episode_id", "nunique"),
            winner_observations=("winner_observation_count", "sum"),
        )
        .reset_index()
    )
    team_card_counts["card_id"] = team_card_counts["card_name"].map(parse_card_id).astype("Int64")
    team_card_counts["winner_obs_share"] = team_card_counts["winner_observations"] / team_card_counts["observations"].replace(0, np.nan)
    if len(card_master):
        team_card_counts = team_card_counts.merge(card_master, on="card_id", how="left")
    team_card_counts["display_name_ja"] = team_card_counts.apply(
        lambda row: row.get("name_ja") if pd.notna(row.get("name_ja")) else card_display_name(row["card_name"], 80),
        axis=1,
    )
    needed_ids = team_card_counts.sort_values(["episodes", "observations"], ascending=False)["card_id"].dropna().astype(int).head(72).tolist()
    team_image_paths = extract_official_card_images(needed_ids)
    team_card_counts["official_image_path"] = team_card_counts["card_id"].map(lambda x: team_image_paths.get(int(x)) if pd.notna(x) else None)
    team_card_counts = team_card_counts.merge(
        strong_teams[["team", "games", "wins", "win_rate", "mean_episode_steps"]],
        on="team",
        how="left",
    ).sort_values(["win_rate", "team", "episodes", "observations"], ascending=[False, True, False, False])

    display_ja(strong_teams[["team", "games", "wins", "losses", "win_rate", "mean_episode_steps"]], head=20)
    display_ja(team_card_counts[[
        "team", "win_rate", "games", "card_id", "display_name_ja", "name_en",
        "episodes", "observations", "winner_obs_share", "official_image_path",
    ]], head=120)




In [ ]:
def render_strong_team_card_gallery(team_cards, teams, cards_per_team=8):
    if not len(team_cards) or not len(teams):
        return
    sections = []
    for team_row in teams.itertuples(index=False):
        team = getattr(team_row, "team")
        subset = team_cards[team_cards["team"] == team].sort_values(["episodes", "observations"], ascending=False).head(cards_per_team)
        if subset.empty:
            continue
        cards = []
        for row in subset.itertuples(index=False):
            img_path = getattr(row, "official_image_path", None)
            img = html_img(img_path, 104) if img_path else '<div style="width:104px;height:146px;border-radius:8px;background:#eef2f7;display:flex;align-items:center;justify-content:center;color:#64748b;font-size:11px;text-align:center;">公式画像<br>未抽出</div>'
            cards.append(f"""
            <div style="display:flex;gap:10px;border:1px solid #dbe3ee;border-radius:8px;padding:8px;background:#ffffff;">
              <div>{img}</div>
              <div style="min-width:0;">
                <div style="font-weight:700;color:#0f172a;font-size:13px;line-height:1.35;">{escape(str(getattr(row, "display_name_ja", "")))}</div>
                <div style="color:#64748b;font-size:11px;margin-top:2px;">ID {escape(str(getattr(row, "card_id", "")))} / {escape(str(getattr(row, "name_en", "")))}</div>
                <div style="margin-top:8px;color:#334155;font-size:11px;">見えた対戦 {getattr(row, "episodes", 0):,}</div>
                <div style="color:#334155;font-size:11px;">観測 {getattr(row, "observations", 0):,}</div>
              </div>
            </div>
            """)
        sections.append(f"""
        <section style="border:1px solid #cbd5e1;border-radius:8px;padding:12px;background:#f8fafc;margin:0 0 14px 0;">
          <div style="font-weight:800;color:#0f172a;font-size:16px;margin-bottom:4px;">{escape(str(team))}</div>
          <div style="color:#475569;font-size:12px;margin-bottom:10px;">勝率 {getattr(team_row, "win_rate", 0):.1%} / 勝ち {getattr(team_row, "wins", 0):,} / 試合 {getattr(team_row, "games", 0):,} / 平均ステップ {getattr(team_row, "mean_episode_steps", 0):.1f}</div>
          <div style="display:grid;grid-template-columns:repeat(auto-fit,minmax(250px,1fr));gap:10px;">{''.join(cards)}</div>
        </section>
        """)
    html = "\n".join(sections)
    try:
        from IPython.display import HTML, display as ipy_display
        ipy_display(HTML(html))
    except Exception:
        print("勝率上位チームのカードギャラリーはNotebookとして実行すると表示されます。")


if "team_card_counts" in globals() and "strong_teams" in globals():
    render_strong_team_card_gallery(team_card_counts, strong_teams, cards_per_team=8)




In [ ]:
if HAS_PLOT and "team_card_counts" in globals() and len(team_card_counts):
    top_team_names = strong_teams["team"].tolist()
    top_card_names = (
        team_card_counts.groupby("display_name_ja")["episodes"]
        .sum()
        .sort_values(ascending=False)
        .head(18)
        .index
        .tolist()
    )
    heat_src = team_card_counts[
        team_card_counts["team"].isin(top_team_names)
        & team_card_counts["display_name_ja"].isin(top_card_names)
    ].copy()
    heat_src["team_label"] = heat_src["team"].map(lambda x: safe_plot_label(x, 18))
    heat_src["card_label"] = heat_src["display_name_ja"].map(lambda x: safe_plot_label(x, 24))
    heat = heat_src.pivot_table(index="team_label", columns="card_label", values="episodes", aggfunc="sum", fill_value=0)
    if len(heat):
        fig, ax = plt.subplots(figsize=(13.5, max(4.8, len(heat) * 0.62)))
        sns.heatmap(heat, annot=True, fmt=".0f", cmap="YlGnBu", linewidths=0.6, linecolor="#f1f5f9", ax=ax)
        ax.set_xlabel("見えているカード")
        ax.set_ylabel("勝率上位チーム")
        finish_plot("20_strong_team_visible_card_heatmap.png", "勝率上位チームの使用カード傾向", "数字はそのカードが見えた対戦数。公開観測のみで完全なデッキリストではありません")




## ログとイベント文

ルールイベントを追うときはログが近道です。
まず頻出ログを見てから、特定カード名や失敗パターンを検索します。




In [ ]:
logs_df = pd.DataFrame([{"log": k, "count": v} for k, v in log_counter.most_common(200)])
display_ja(logs_df, head=80)




In [ ]:
if HAS_PLOT and len(logs_df):
    top_logs_plot = logs_df.head(25).sort_values("count")
    labels = top_logs_plot["log"].str.slice(0, 80)
    fig, ax = plt.subplots(figsize=(11.5, 7.5))
    sns.barplot(x=top_logs_plot["count"], y=labels, ax=ax, color=PALETTE["gold"], orient="h")
    ax.set_xlabel("件数")
    ax.set_ylabel("")
    finish_plot("08_top_log_events.png", "頻出する生リプレイログ", "イベント種別IDやゾーン移動をデコードするための地図")




In [ ]:
SEARCH_TERMS = ["Iono", "ナンジャモ", "Lucario", "ルカリオ", "Dragapult", "ドラパルト", "Abomasnow", "ユキノオー", "deck", "discard", "knock"]
if len(logs_df):
    hits = []
    for term in SEARCH_TERMS:
        mask = logs_df["log"].str.contains(term, case=False, na=False, regex=False)
        if mask.any():
            tmp = logs_df[mask].copy()
            tmp.insert(0, "term", term)
            hits.append(tmp)
    if hits:
        display(pd.concat(hits).drop_duplicates().head(120))
    else:
        print("設定済み検索語は上位ログ内で見つかりませんでした。")




## 勝者側と敗者側の比較

ここは差分をざっくり見る場所です。
因果を主張するためではなく、次にどのリプレイやIDを詳しく見るかを決めるために使います。




In [ ]:
if len(actions_df):
    winner_action = actions_df.groupby(["is_winner_player", "action_kind_ja"]).size().reset_index(name="count")
    totals = winner_action.groupby("is_winner_player")["count"].transform("sum")
    winner_action["share"] = winner_action["count"] / totals
    pivot = winner_action.pivot(index="action_kind_ja", columns="is_winner_player", values="share").fillna(0)
    if True in pivot.columns and False in pivot.columns:
        pivot["winner_minus_loser_share"] = pivot[True] - pivot[False]
    display_ja(pivot.sort_values("winner_minus_loser_share" if "winner_minus_loser_share" in pivot else pivot.columns[0], ascending=False).reset_index())




In [ ]:
if len(cards_df):
    wc = cards_df.groupby(["card_name", "is_winner_player"]).agg(obs=("observations", "sum"), episodes=("episode_id", "nunique")).reset_index()
    obs_pivot = wc.pivot(index="card_name", columns="is_winner_player", values="episodes").fillna(0)
    obs_pivot.columns = [f"episodes_seen_winner_{c}" for c in obs_pivot.columns]
    true_col = "episodes_seen_winner_True"
    false_col = "episodes_seen_winner_False"
    if true_col not in obs_pivot: obs_pivot[true_col] = 0
    if false_col not in obs_pivot: obs_pivot[false_col] = 0
    obs_pivot["winner_minus_loser_episode_seen"] = obs_pivot[true_col] - obs_pivot[false_col]
    obs_pivot["display_name_ja"] = [card_display_name(idx, 80) for idx in obs_pivot.index]
    display_ja(obs_pivot.sort_values("winner_minus_loser_episode_seen", ascending=False).reset_index(), head=60)
    display_ja(obs_pivot.sort_values("winner_minus_loser_episode_seen", ascending=True).reset_index(), head=60)




In [ ]:
if HAS_PLOT and len(cards_df) and "obs_pivot" in globals():
    delta_plot = pd.concat([
        obs_pivot.sort_values("winner_minus_loser_episode_seen", ascending=False).head(15),
        obs_pivot.sort_values("winner_minus_loser_episode_seen", ascending=True).head(15),
    ]).drop_duplicates().sort_values("winner_minus_loser_episode_seen")
    fig, ax = plt.subplots(figsize=(10.5, 8.2))
    vals = delta_plot["winner_minus_loser_episode_seen"]
    colors = [PALETTE["green"] if v >= 0 else PALETTE["red"] for v in vals]
    ax.barh(delta_plot["display_name_ja"].map(lambda x: safe_plot_label(x, 34)), vals, color=colors, alpha=0.84)
    ax.axvline(0, color="#333333", linewidth=1)
    ax.set_xlabel("勝者側で見えた対戦数 - 敗者側で見えた対戦数")
    finish_plot("09_visible_card_winner_loser_delta.png", "勝者側/敗者側に偏って見えるカード", "偏りは記述統計です。因果ではなく、確認するリプレイ選びに使います")




## 多角的な戦略ビュー

同じリプレイサンプルを、いくつかの角度から見直します。

- チーム勝率と対戦速度
- 決着タイプ別の試合長
- 勝者側/敗者側に偏る行動カテゴリ
- 見えているカードIDの共起
- 生ログのイベント種別構造

いずれも記述的なEDAです。因果を言い切るためではなく、深掘りするリプレイや行動を選ぶために使います。




In [ ]:
if HAS_PLOT and "team_summary" in globals() and len(team_summary):
    plot = team_summary.copy()
    plot = plot[plot["games"] >= max(1, min(3, plot["games"].quantile(0.50)))].head(60)
    plot = plot.rename(columns={"games": "試合数", "wins": "勝利数", "mean_episode_steps": "平均対戦ステップ数", "win_rate": "勝率"})
    fig, ax = plt.subplots(figsize=(10.8, 6.2))
    sns.scatterplot(
        data=plot,
        x="平均対戦ステップ数",
        y="勝率",
        size="試合数",
        hue="勝利数",
        palette="viridis",
        sizes=(60, 520),
        alpha=0.78,
        ax=ax,
    )
    for _, row in plot.sort_values("勝利数", ascending=False).head(12).iterrows():
        ax.text(row["平均対戦ステップ数"] + 1, row["勝率"], safe_plot_label(row["team"], 18), fontsize=8, color="#334155")
    ax.axhline(0.5, color=PALETTE["gray"], linestyle="--", linewidth=1)
    ax.set_xlabel("平均対戦ステップ数")
    ax.set_ylabel("勝率")
    ax.set_ylim(-0.04, 1.04)
    ax.legend(frameon=False, bbox_to_anchor=(1.02, 1), loc="upper left")
    finish_plot("12_team_strength_speed_scatter.png", "チーム勝率と対戦速度", "速く勝つクラスタは序盤圧力、遅く勝つクラスタはリソース循環の候補")




In [ ]:
def episode_outcome_label(row):
    r0 = pd.to_numeric(row.get("reward_0"), errors="coerce")
    r1 = pd.to_numeric(row.get("reward_1"), errors="coerce")
    s0 = str(row.get("status_0", ""))
    s1 = str(row.get("status_1", ""))
    statuses = f"{s0} {s1}"
    if pd.notna(r0) and pd.notna(r1) and sorted([float(r0), float(r1)]) == [-1.0, 1.0]:
        return "通常決着"
    if pd.notna(r0) and pd.notna(r1) and float(r0) == 0.0 and float(r1) == 0.0:
        return "引き分け"
    if "TIMEOUT" in statuses:
        return "時間切れ/未完了"
    if "ERROR" in statuses or "INVALID" in statuses:
        return "エラー/無効"
    return "その他"


if HAS_PLOT and len(episodes_df):
    plot = episodes_df.copy()
    plot["決着タイプ"] = plot.apply(episode_outcome_label, axis=1)
    order = ["通常決着", "引き分け", "時間切れ/未完了", "エラー/無効", "その他"]
    order = [x for x in order if x in set(plot["決着タイプ"])]
    fig, ax = plt.subplots(figsize=(9.6, 5.2))
    palette = {
        "通常決着": PALETTE["green"],
        "引き分け": PALETTE["gray"],
        "時間切れ/未完了": PALETTE["orange"],
        "エラー/無効": PALETTE["red"],
        "その他": PALETTE["purple"],
    }
    sns.boxplot(data=plot, x="決着タイプ", y="steps", order=order, hue="決着タイプ", palette=palette, fliersize=2.5, ax=ax, legend=False)
    sample_parts = []
    for _, group in plot.groupby("決着タイプ", group_keys=False):
        sample_parts.append(group.sample(min(len(group), 120), random_state=RANDOM_SEED))
    sample = pd.concat(sample_parts, ignore_index=True) if sample_parts else plot.head(0)
    sns.stripplot(data=sample, x="決着タイプ", y="steps", order=order, color="#1f2937", alpha=0.22, size=2.4, ax=ax)
    counts = plot["決着タイプ"].value_counts()
    top = plot["steps"].max()
    for pos, label in enumerate(order):
        ax.text(pos, top * 1.01, f"n={counts.get(label, 0):,}", ha="center", va="bottom", fontsize=8, color="#475569")
    ax.set_xlabel("")
    ax.set_ylabel("対戦ステップ数")
    finish_plot("13_result_length_violin.png", "決着タイプ別の対戦長", "勝者/敗者は同じ試合長になるため、引き分け・時間切れ・エラーを分けて確認")

    zoom_limit = float(plot["steps"].quantile(0.99))
    zoom_limit = max(120.0, min(500.0, math.ceil(zoom_limit / 10.0) * 10.0))
    fig, ax = plt.subplots(figsize=(9.6, 5.2))
    sns.boxplot(
        data=plot,
        x="決着タイプ",
        y="steps",
        order=order,
        hue="決着タイプ",
        palette=palette,
        showfliers=False,
        ax=ax,
        legend=False,
    )
    sample_zoom = sample[sample["steps"] <= zoom_limit]
    sns.stripplot(data=sample_zoom, x="決着タイプ", y="steps", order=order, color="#1f2937", alpha=0.22, size=2.4, ax=ax)
    for pos, label in enumerate(order):
        above = int((plot.loc[plot["決着タイプ"] == label, "steps"] > zoom_limit).sum())
        note = f"n={counts.get(label, 0):,}"
        if above:
            note += f"\n>{int(zoom_limit)}: {above:,}"
        ax.text(pos, zoom_limit * 1.01, note, ha="center", va="bottom", fontsize=8, color="#475569")
    ax.set_ylim(0, zoom_limit * 1.12)
    ax.set_xlabel("")
    ax.set_ylabel("対戦ステップ数")
    finish_plot("13b_result_length_no_outlier_zoom.png", "決着タイプ別の対戦長（外れ値なし）", f"上位1%付近を外して通常帯を拡大。{int(zoom_limit)}ステップ超は注記")




In [ ]:
if HAS_PLOT and len(actions_df):
    skew_base = actions_df.groupby(["action_kind_ja", "is_winner_player"], dropna=False).size().reset_index(name="count")
    totals = skew_base.groupby("is_winner_player")["count"].transform("sum")
    skew_base["share"] = skew_base["count"] / totals
    skew = skew_base.pivot(index="action_kind_ja", columns="is_winner_player", values="share").fillna(0)
    action_totals = actions_df["action_kind_ja"].value_counts()
    skew["total_actions"] = action_totals.reindex(skew.index).fillna(0)
    if True in skew.columns and False in skew.columns:
        skew["winner_minus_loser_share"] = skew[True] - skew[False]
        skew = skew[skew["total_actions"] >= max(MIN_ACTION_COUNT, skew["total_actions"].quantile(0.35))]
        skew = skew.sort_values("winner_minus_loser_share")
        fig, ax = plt.subplots(figsize=(9.8, max(4.8, len(skew) * 0.42)))
        colors = [PALETTE["green"] if v >= 0 else PALETTE["red"] for v in skew["winner_minus_loser_share"]]
        labels = [f"{safe_plot_label(idx, 26)} ({int(row['total_actions']):,})" for idx, row in skew.iterrows()]
        ax.barh(labels, skew["winner_minus_loser_share"], color=colors, alpha=0.84)
        ax.axvline(0, color="#333333", linewidth=1)
        ax.margins(x=0.12)
        ax.set_xlabel("勝者側の構成比 - 敗者側の構成比")
        ax.set_ylabel("行動カテゴリ")
        finish_plot("14_action_id_winner_skew.png", "勝者側/敗者側に偏る行動カテゴリ", "正なら勝者側で多い。括弧内はそのカテゴリの総レコード数")




In [ ]:
if HAS_PLOT and len(cards_df):
    top_cards = card_counts.head(18)["card_name"].tolist() if "card_counts" in globals() else cards_df["card_name"].value_counts().head(18).index.tolist()
    ep_card = cards_df[cards_df["card_name"].isin(top_cards)].drop_duplicates(["episode_id", "card_name"])
    mat = pd.crosstab(ep_card["episode_id"], ep_card["card_name"]).clip(upper=1)
    if mat.shape[1] >= 2:
        mat = mat.rename(columns={c: card_display_name(c, 24) for c in mat.columns})
        co = mat.T.dot(mat)
        np.fill_diagonal(co.values, 0)
        fig, ax = plt.subplots(figsize=(10.8, 8.8))
        sns.heatmap(co, cmap="mako", linewidths=0.35, linecolor="white", square=True, ax=ax, cbar_kws={"label": "同時に見えた対戦数"})
        ax.set_xlabel("")
        ax.set_ylabel("")
        finish_plot("15_visible_card_cooccurrence_heatmap.png", "見えているカードの共起ヒートマップ", "同じリプレイで一緒に見えたカード。アーキタイプの署名候補として使います")




In [ ]:
def parse_log_dict(text):
    try:
        value = ast.literal_eval(text)
    except Exception:
        return {}
    return value if isinstance(value, dict) else {}

if len(logs_df):
    log_struct = []
    for row in logs_df.itertuples(index=False):
        d = parse_log_dict(row.log)
        raw_type = d.get("type", "unknown")
        type_label = LOG_TYPE_JA.get(raw_type, f"種別{raw_type}") if raw_type != "unknown" else "不明"
        log_struct.append({
            "type": raw_type,
            "type_ja": type_label,
            "fromArea": d.get("fromArea", "none"),
            "toArea": d.get("toArea", "none"),
            "count": row.count,
        })
    log_struct_df = pd.DataFrame(log_struct)
    display_ja(log_struct_df, head=80)

    if HAS_PLOT and len(log_struct_df):
        type_counts = log_struct_df.groupby("type_ja", dropna=False)["count"].sum().sort_values(ascending=False).head(20).sort_values()
        fig, ax = plt.subplots(figsize=(9.5, 5.2))
        sns.barplot(x=type_counts.values, y=type_counts.index.astype(str), ax=ax, color=PALETTE["purple"], orient="h")
        ax.set_xlabel("上位生ログ行からの重み付き件数")
        ax.set_ylabel("生ログ種別")
        finish_plot("16_log_type_distribution.png", "生ログイベント種別", "IDだけでなく、主な意味を日本語ラベルで表示")

        zone = log_struct_df[(log_struct_df["fromArea"] != "none") | (log_struct_df["toArea"] != "none")]
        if len(zone):
            zone = zone.copy()
            zone["移動元エリア"] = zone["fromArea"].map(area_label)
            zone["移動先エリア"] = zone["toArea"].map(area_label)
            zone_mat = zone.pivot_table(index="移動元エリア", columns="移動先エリア", values="count", aggfunc="sum", fill_value=0)
            preferred_order = ["なし/不明"] + list(AREA_JA.values())
            row_order = [x for x in preferred_order if x in zone_mat.index] + [x for x in zone_mat.index if x not in preferred_order]
            col_order = [x for x in preferred_order if x in zone_mat.columns] + [x for x in zone_mat.columns if x not in preferred_order]
            zone_mat = zone_mat.loc[row_order, col_order]
            fig, ax = plt.subplots(figsize=(10.2, 6.4))
            sns.heatmap(zone_mat, cmap="rocket_r", annot=True, fmt=".0f", linewidths=0.4, linecolor="white", ax=ax, cbar_kws={"label": "移動イベント数"})
            ax.set_xlabel("移動先エリア")
            ax.set_ylabel("移動元エリア")
            finish_plot("17_log_zone_transition_heatmap.png", "公開ログのカード移動元→移動先", "数字は頻出ログから復元した移動イベント数。山札→手札はサーチ/ドロー候補")




## 目視レビュー候補の対戦

目視リプレイ、模倣学習、手作業の方策デバッグに回す対戦候補です。




In [ ]:
if len(episodes_df):
    review_queue = episodes_df.copy()
    review_queue["abs_reward_gap"] = (pd.to_numeric(review_queue["reward_0"], errors="coerce") - pd.to_numeric(review_queue["reward_1"], errors="coerce")).abs()
    review_queue["reason"] = np.select(
        [review_queue["steps"] <= review_queue["steps"].quantile(0.10), review_queue["steps"] >= review_queue["steps"].quantile(0.90)],
        ["短い試合:序盤方針確認", "長い試合:リソース循環確認"],
        default="通常の上位対戦"
    )
    cols = ["episode_id", "date", "reason", "steps", "team_0", "team_1", "reward_0", "reward_1", "path"]
    display_ja(review_queue.sort_values(["reason", "steps"]).loc[:, cols], head=80)




In [ ]:
if HAS_PLOT and len(episodes_df) and "review_queue" in globals():
    short_games = review_queue.nsmallest(12, "steps").copy()
    long_games = review_queue.nlargest(12, "steps").copy()
    def review_label(row):
        left = safe_plot_label(row.get("team_0", ""), 13)
        right = safe_plot_label(row.get("team_1", ""), 13)
        return f"{int(row['episode_id'])}: {left} vs {right}"
    short_games["label"] = short_games.apply(review_label, axis=1)
    long_games["label"] = long_games.apply(review_label, axis=1)
    fig, axes = plt.subplots(1, 2, figsize=(14.2, 6.2), sharex=False)
    panels = [
        (axes[0], short_games.sort_values("steps", ascending=True), PALETTE["orange"], "短い試合候補", "序盤の展開崩れ・早期決着を見る"),
        (axes[1], long_games.sort_values("steps", ascending=True), PALETTE["blue"], "長い試合候補", "山札/リソース循環や停滞を見る"),
    ]
    for panel_ax, data, color, title, subtitle in panels:
        panel_ax.barh(data["label"], data["steps"], color=color, alpha=0.82)
        panel_ax.set_title(f"{title}\n{subtitle}", fontsize=10, color="#334155")
        panel_ax.set_xlabel("ステップ数")
        panel_ax.set_ylabel("")
        for i, value in enumerate(data["steps"]):
            panel_ax.text(value + max(1, data["steps"].max() * 0.01), i, f"{int(value):,}", va="center", fontsize=8, color="#475569")
    finish_plot("10_review_queue_step_outliers.png", "目視レビュー候補: 短すぎる/長すぎる対戦", "短い=序盤方針、長い=リソース循環や停滞。episode_idからリプレイを追う")




## EDAテーブルの出力

Kaggleの出力タブからCSVと画像をダウンロードできます。
後続ノートブックに渡しやすいよう、CSVは大きくなりすぎない範囲にまとめています。




In [ ]:
OUT_DIR = Path("/kaggle/working/ptcg_official_top_episodes_eda_ja") if Path("/kaggle/working").exists() else Path("ptcg_official_top_episodes_eda_ja")
OUT_DIR.mkdir(parents=True, exist_ok=True)

exports = {
    "episodes.csv": episodes_df,
    "players.csv": players_df,
    "actions_sample.csv": actions_df.head(200000),
    "visible_cards_jp.csv": card_counts if "card_counts" in globals() else pd.DataFrame(),
    "board_action_counts.csv": board_action_counts if "board_action_counts" in globals() else pd.DataFrame(),
    "action_correlation.csv": action_correlation if "action_correlation" in globals() else pd.DataFrame(),
    "strong_team_visible_cards.csv": team_card_counts if "team_card_counts" in globals() else pd.DataFrame(),
    "top_logs.csv": logs_df if "logs_df" in globals() else pd.DataFrame(),
    "review_queue.csv": review_queue if "review_queue" in globals() else pd.DataFrame(),
}
for name, df in exports.items():
    path = OUT_DIR / name
    df.to_csv(path, index=False)
    print(path, df.shape)

summary = {
    "max_episodes": MAX_EPISODES,
    "date_filter": DATE_FILTER,
    "episodes": int(len(episodes_df)),
    "players": int(len(players_df)),
    "actions": int(len(actions_df)),
    "visible_card_observations": int(cards_df["observations"].sum()) if len(cards_df) and "observations" in cards_df else int(len(cards_df)),
    "visible_card_rows": int(len(cards_df)),
    "manifest_rows": int(len(manifest)),
    "card_master_rows": int(len(card_master)),
    "official_card_pdf": str(JP_CARD_PDF) if JP_CARD_PDF else None,
    "official_card_images": int(len(list(CARD_IMAGE_DIR.glob("*.png")))),
    "board_action_patterns": int(len(board_action_counts)) if "board_action_counts" in globals() else 0,
    "action_correlation_rows": int(len(action_correlation)) if "action_correlation" in globals() else 0,
    "card_image_dir": str(CARD_IMAGE_DIR),
    "matplotlib_japanese_font": str(JAPANESE_FONT),
    "output_dir": str(OUT_DIR),
    "plot_dir": str(PLOT_DIR),
    "plots": sorted(p.name for p in PLOT_DIR.glob("*.png")),
}
(OUT_DIR / "summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary




## 次に見るべきポイント

出力テーブルは、この順番で見るのが分かりやすいです。

1. 見直し候補テーブルから、短い負け試合や長い勝ち試合を選んで目視する。
2. 可視カード一覧で、よく出るカードの組み合わせやアーキタイプの気配を見る。
3. `board_action_counts.csv` で、盤面条件ごとに実際に選ばれたカード/ワザ/番終了を見る。
4. `action_correlation.csv` で、勝者側に寄りやすい行動を特徴量候補として見る。
5. 良さそうな傾向は、新しい日別ログでも再確認してから戦略コードやデッキに反映する。

注意: リプレイ観測は完全な非公開情報ではありません。
手札や正確な山札内容は、別途公開情報から再構成しない限り「部分的にしか見えていない」と扱います。
実戦AIに落とす場合は、全行動から直接選ぶのではなく、その局面の合法手候補を同じ特徴量で順位付けします。
